# M115 — TP3
## Grain, modèle et ETL dépendant du modèle
### Notebook étudiant à compléter · Kaggle · Python/Pandas
**Pr. AMINA ABOULMIRA · ENSAB**

| Identification du binôme | À renseigner |
|---|---|
| Étudiant(e) 1 — nom et prénom |AKOUAR Fatima Zouhra |
| Étudiant(e) 2 — nom et prénom |ABGAR Salma |
| Groupe / classe | Master IAA |
| Date de réalisation | 21/10/2026 |
| Lien ou titre du notebook Kaggle | … |

**Mission :** transformer les données préparées L2 en un modèle décisionnel cohérent, alimenté et contrôlé : **L3**.

**Durée : 4 h.** Travail en binômes. Alternez les rôles : l’un écrit et l’autre vérifie, puis échangez. Les questions complémentaires renforcent les notions étudiées jusqu’à C4 ; elles n’ajoutent pas une évaluation ou un second projet.

Ce notebook est un **support de travail**, pas un corrigé : rédigez vos réponses dans les cellules Markdown, écrivez le code dans les cellules prévues et conservez vos sorties.

## Notre parcours du TP3 — illustration à ajouter

<table><tr><td style="border:2px dashed #42A9D7;padding:45px;text-align:center">
<strong>EMPLACEMENT RÉSERVÉ À L’IMAGE DU PARCOURS TP3</strong><br><br>
L’enseignante ajoutera ici son illustration des étapes.<br>
L2 → grain → modèle → dimensions → rattachement des clés → fait → contrôles → L3.
</td></tr></table>

<!-- Remplacer ce bloc par votre illustration, ou insérer une cellule image juste ici. Aucun lien d’image externe n’est imposé. -->

## Avant de commencer

Les **CSV seront fournis**. L’entrée principale est la source préparée du TP2, normalement nommée `TP2_L2_SOURCE_PREPAREE.csv`. Les autres fichiers L2 servent de traces et de contrôles. N’utilisez pas le brut `Online_Retail.csv` comme entrée de ce TP.

1. Ajouter les CSV fournis aux données du notebook Kaggle.
2. Exécuter la cellule d’environnement ci-dessous, puis identifier le chemin exact du fichier préparé.
3. Travailler dans l’ordre, avec les noms de DataFrames imposés.
4. Ne pas refaire le filtrage ou le dédoublonnage du TP2. Si L2 pose problème, documenter l’écart avant de poursuivre.
5. Répondre avant de coder, puis confronter la réponse aux sorties réelles.

**Modèle MVP :** `FactVente + DimDate + DimProduit + DimClient + DimPays`.

**Règles maintenues :** ventes positives de produits commerciaux ; client absent conservé ; pays disponible ; aucune catégorie inventée ; `InvoiceNo` reste dans le fait ; aucun dédoublonnage sur `InvoiceNo + StockCode` seuls.

Power BI n’est pas requis ici. **Le prochain TP portera sur l’OLAP**, à partir de L3.

### Repères de validation
- **CP3 — Grain :** expliquer exactement ce que représente une ligne.
- **CP4 — Modèle :** justifier faits, dimensions, mesures, clés et relations.
- **CP5 — Alimentation :** prouver que le chargement conserve les ventes et respecte les références.

### Temps proposé, réflexion et contrôles compris
| Étapes | Travail | Temps |
|---|---|---:|
| 1–4 | Entrée, grain, mesures, modèle | 65 min |
| 5–8 | Dimensions et client inconnu | 65 min |
| 9–10 | Lookups et fait | 45 min |
| 11–13 | Contrôles, architecture et débogage | 50 min |
| 14 | Export L3 et bilan | 15 min |
| **Total** | | **240 min** |

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

# Dossier de travail de Jupyter Notebook
DOSSIER_ENTREE = Path.cwd()
DOSSIER_SORTIE = Path.cwd()

# Rechercher les fichiers CSV disponibles
csv_disponibles = sorted(DOSSIER_ENTREE.glob("*.csv"))

print("CSV disponibles :")

for chemin in csv_disponibles:
    print(chemin)

CSV disponibles :
C:\Users\pc\customer_shopping_data.csv
C:\Users\pc\data.csv
C:\Users\pc\diabetes.csv
C:\Users\pc\heart_data.csv
C:\Users\pc\multiple_linear_regression_dataset.csv
C:\Users\pc\retail_sales_dataset.csv
C:\Users\pc\retail_sales_dataset_augmented.csv
C:\Users\pc\Salary_Data.csv
C:\Users\pc\sales_data_ulmann.csv
C:\Users\pc\TP2_L2_SOURCE_PREPAREE.csv


## Contrat des noms à respecter

| Nom dans ce notebook | Contenu attendu |
|---|---|
| `df_prep` | CSV L2 chargé ; référence à conserver sans modification |
| `df` | Copie de travail pour les colonnes dérivées du modèle |
| `dim_date` | Dimension au grain du jour |
| `dim_produit` | Couples code / description normalisée distincts |
| `dim_client` | Clients identifiés et membre « Client inconnu » |
| `dim_pays` | Pays distincts de la source |
| `df_rattache` | Ventes enrichies des quatre clés dimensionnelles |
| `fact_vente` | Fait final exporté sous le nom FactVente |
| `controles_l3` | Tableau des preuves calculées |

**Noms des tables exportées :** DimDate, DimProduit, DimClient, DimPays et FactVente. Les noms Python en minuscules ne changent pas les noms du modèle.

**Mesures du fait :** `QuantiteVendue`, `PrixUnitaire`, `MontantLigne`. Les deux premières correspondent à `Quantity` et `UnitPrice` de L2 ; ce mapping doit être documenté.

# ÉTAPE 01 — Vérifier l’entrée L2 et situer la chaîne BI
**Durée indicative : 15 min · Objectif :** Comprendre les fichiers reçus avant de les transformer.



## Questions du TP
1. Comment passer d’une source préparée à un modèle multidimensionnel cohérent et contrôlable ?
2. Quelles structures faut-il construire pour analyser les ventes selon le temps, le produit, le client et le pays sans changer le sens des données ?

## Question complémentaire — prendre du recul
Dans la chaîne sources → staging → préparation / intégration → entrepôt → exploitation, où se situe L2 ? Quelles opérations du TP2 ne faut-il pas refaire ici ?

### Réponses du binôme — à compléter

| Fichier CSV fourni | Rôle / usage dans TP3 |
|---|---|
| TP2_L2_SOURCE_PREPAREE.csv | Source principale contenant les ventes préparées. |


**Colonnes disponibles et types observés :** 
Après l'importation et l'analyse du fichier TP2_L2_SOURCE_PREPAREE.csv, nous avons constaté que notre source L2 contient 522 364 lignes et 28 colonnes.

Elle contient les huit colonnes métier originales :
| Colonne      | Type observé | Description             |
|--------------|--------------|-------------------------|
| InvoiceNo    | int64        | Numéro de facture       |
| StockCode    | object       | Code du produit         |
| Description  | object       | Description du produit  |
| Quantity     | int64        | Quantité vendue         |
| InvoiceDate  | object       | Date de la transaction  |
| UnitPrice    | float64      | Prix unitaire           |
| CustomerID   | float64      | Identifiant du client   |
| Country      | object       | Pays du client          |

La source contient également des colonnes techniques et dérivées permettant d'assurer la traçabilité et la préparation des données :
- Colonnes de traçabilité : SourceIndex, SourceRow, SourceFile.
- Colonnes normalisées : StockCode_norm, Description_norm, Country_norm, InvoiceNo_norm, CustomerID_norm.
- Colonnes numériques : Quantity_num, UnitPrice_num.
- Colonnes de contrôle : Quantity_echec, UnitPrice_echec, InvoiceDate_echec.
- Colonnes de classification : TypeLigne, OrigineClassification, Annulation, MotifsExclusion, InclureFactVente.
- Colonne calculée : MontantLigne.
- 
**Colonnes manquantes ou différences avec le contrat :** 
Les huit colonnes métier originales sont toutes présentes. Les colonnes techniques et dérivées nécessaires sont également disponibles.
Cependant, nous avons remarqué quelques particularités concernant les types :
- InvoiceNo est de type numérique (int64), alors qu'il représente un identifiant et devra être traité comme du texte dans le modèle.
- CustomerID est de type float64 en raison de la présence de valeurs manquantes. Il faudra préserver son caractère d'identifiant.
- InvoiceDate est de type object, mais une colonne dérivée InvoiceDate_dt contient déjà les dates converties.
- La colonne MotifsExclusion est entièrement vide dans cette source.
Nous avons également constaté que la colonne CustomerID contient 131 355 valeurs manquantes, soit environ 25,15 % des lignes.

**Notre justification :**  
Nous avons choisi TP2_L2_SOURCE_PREPAREE.csv comme source principale, car elle contient les données déjà préparées et contrôlées lors du TP2 (522 364 lignes et 28 colonnes).
Cette source nous permet d'éviter de refaire le nettoyage et le dédoublonnage. Nous allons l'utiliser pour construire FactVente et les quatre dimensions : DimDate, DimProduit, DimClient et DimPays, afin d'analyser les ventes selon différents axes. Les clients sans identifiant seront conservés sous la catégorie « Client inconnu ».

**Point à vérifier / difficulté rencontrée :**  
Nous avons constaté quelques différences dans les types des identifiants : InvoiceNo est numérique et CustomerID est décimal. Une conversion adaptée sera donc nécessaire.
Nous avons également relevé 131 355 valeurs manquantes dans CustomerID. Ces clients seront conservés et rattachés à « Client inconnu ». Enfin, les colonnes dérivées nécessaires (InvoiceDate_dt, Quantity_num, UnitPrice_num et MontantLigne) sont déjà disponibles dans la source.

In [2]:

# ============================================================
# ÉTAPE 01 : VÉRIFICATION DE LA SOURCE L2
# TP3 - Construction du modèle multidimensionnel
# ============================================================

# 1. Importation des bibliothèques nécessaires

from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display


# 2. Définition des dossiers de travail
# Path.cwd() permet de récupérer le dossier courant de Jupyter.

DOSSIER_ENTREE = Path.cwd()
DOSSIER_SORTIE = Path.cwd()

print("Dossier de travail :", DOSSIER_ENTREE)


# 3. Recherche des fichiers CSV disponibles

csv_disponibles = sorted(DOSSIER_ENTREE.glob("*.csv"))

print("\nFichiers CSV disponibles :")

for chemin in csv_disponibles:
    print(chemin.name)


# 4. Identification automatique du fichier préparé TP2

fichiers_l2 = [
    chemin for chemin in csv_disponibles
    if chemin.name == "TP2_L2_SOURCE_PREPAREE.csv"
]

# Vérifier si le fichier existe

if len(fichiers_l2) == 0:
    raise FileNotFoundError(
        "Le fichier TP2_L2_SOURCE_PREPAREE.csv est introuvable."
    )

# Récupérer le chemin exact du fichier

CHEMIN_L2 = fichiers_l2[0]

print("\nChemin du fichier L2 :", CHEMIN_L2)


# 5. Chargement de la source L2
# utf-8-sig correspond à l'encodage des exports TP2.
# Les identifiants sont protégés en texte pour préserver leur format.

df_prep = pd.read_csv(
    CHEMIN_L2,
    encoding="utf-8-sig",
    dtype={
        "InvoiceNo": "string",
        "StockCode": "string",
        "CustomerID": "string"
    }
)

print("\nChargement effectué avec succès !")


# 6. Affichage des dimensions et des colonnes

print("\n===== DIMENSIONS DE LA SOURCE L2 =====")

print("Nombre de lignes :", df_prep.shape[0])
print("Nombre de colonnes :", df_prep.shape[1])

print("\n===== COLONNES DISPONIBLES =====")

print(df_prep.columns.tolist())


# 7. Affichage des types de données

print("\n===== TYPES DES COLONNES =====")

print(df_prep.dtypes)


# 8. Vérification des huit colonnes métier originales

colonnes_originales = [
    "InvoiceNo",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "UnitPrice",
    "CustomerID",
    "Country"
]

print("\n===== CONTRÔLE DES COLONNES ORIGINALES =====")

colonnes_manquantes = [
    col for col in colonnes_originales
    if col not in df_prep.columns
]

for col in colonnes_originales:
    if col in df_prep.columns:
        print(col, ": Présente")
    else:
        print(col, ": Absente")

print("\nColonnes originales manquantes :", colonnes_manquantes)


# 9. Vérification des colonnes techniques et dérivées

colonnes_a_verifier = [
    "SourceIndex",
    "MontantLigne",
    "InvoiceDate_dt",
    "Quantity_num",
    "UnitPrice_num",
    "CustomerID_norm",
    "TypeLigne"
]

print("\n===== CONTRÔLE DES COLONNES TECHNIQUES =====")

for col in colonnes_a_verifier:
    if col in df_prep.columns:
        print(col, ": Présente")
    else:
        print(col, ": Absente")


# 10. Affichage d'un extrait de la source

print("\n===== APERÇU DES DONNÉES =====")

display(df_prep.head())


# 11. Création d'une copie de travail
# df_prep reste la référence originale sans modification.

df = df_prep.copy()

print("\nCopie de travail créée : df")
print("Dimensions de df :", df.shape)


# 12. Calcul des références de la source L2
# Ces valeurs serviront à contrôler la future table FactVente.

nb_lignes_L2 = len(df_prep)

# Calcul du total des quantités à partir de Quantity_num
# si cette colonne existe, sinon à partir de Quantity.

if "Quantity_num" in df_prep.columns:
    total_quantite_L2 = pd.to_numeric(
        df_prep["Quantity_num"],
        errors="coerce"
    ).sum()
else:
    total_quantite_L2 = pd.to_numeric(
        df_prep["Quantity"],
        errors="coerce"
    ).sum()


# Calcul du montant total de référence

if "MontantLigne" in df_prep.columns:
    total_montant_L2 = pd.to_numeric(
        df_prep["MontantLigne"],
        errors="coerce"
    ).sum()
else:
    total_montant_L2 = (
        pd.to_numeric(df_prep["Quantity"], errors="coerce")
        *
        pd.to_numeric(df_prep["UnitPrice"], errors="coerce")
    ).sum()


# 13. Affichage des références L2

print("\n===== RÉFÉRENCES DE LA SOURCE L2 =====")

print("Nombre de lignes :", nb_lignes_L2)
print("Quantité totale :", total_quantite_L2)
print("Montant total :", round(total_montant_L2, 2))


# 14. Création d'un tableau récapitulatif des contrôles

controles_entree = pd.DataFrame({
    "Indicateur": [
        "Nombre de lignes",
        "Nombre de colonnes",
        "Quantité totale",
        "Montant total"
    ],

    "Valeur": [
        nb_lignes_L2,
        df_prep.shape[1],
        total_quantite_L2,
        round(total_montant_L2, 2)
    ]
})

print("\n===== TABLEAU RÉCAPITULATIF =====")

display(controles_entree)


# 15. Vérification finale du contrat

if colonnes_manquantes:
    print("\nATTENTION : certaines colonnes obligatoires sont absentes.")
else:
    print("\nVALIDATION : les huit colonnes métier sont présentes.")

print("\nÉTAPE 01 TERMINÉE.")


Dossier de travail : C:\Users\pc

Fichiers CSV disponibles :
customer_shopping_data.csv
data.csv
diabetes.csv
heart_data.csv
multiple_linear_regression_dataset.csv
retail_sales_dataset.csv
retail_sales_dataset_augmented.csv
Salary_Data.csv
sales_data_ulmann.csv
TP2_L2_SOURCE_PREPAREE.csv

Chemin du fichier L2 : C:\Users\pc\TP2_L2_SOURCE_PREPAREE.csv

Chargement effectué avec succès !

===== DIMENSIONS DE LA SOURCE L2 =====
Nombre de lignes : 522364
Nombre de colonnes : 28

===== COLONNES DISPONIBLES =====
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country', 'SourceIndex', 'SourceRow', 'SourceFile', 'StockCode_norm', 'Description_norm', 'Country_norm', 'InvoiceNo_norm', 'CustomerID_norm', 'Quantity_num', 'Quantity_echec', 'UnitPrice_num', 'UnitPrice_echec', 'InvoiceDate_dt', 'InvoiceDate_echec', 'TypeLigne', 'OrigineClassification', 'Annulation', 'MotifsExclusion', 'InclureFactVente', 'MontantLigne']

===== TYPES DES COLONNES =====
I

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,SourceIndex,SourceRow,...,UnitPrice_num,UnitPrice_echec,InvoiceDate_dt,InvoiceDate_echec,TypeLigne,OrigineClassification,Annulation,MotifsExclusion,InclureFactVente,MontantLigne
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/10 8:26,2.55,17850.0,United Kingdom,0,2,...,2.55,False,2010-12-01 08:26:00,False,PRODUIT_COMMERCIAL,REGLE_PAR_DEFAUT,False,NaN,True,15.30
1,536365,71053,WHITE METAL LANTERN,6,12/1/10 8:26,3.39,17850.0,United Kingdom,1,3,...,3.39,False,2010-12-01 08:26:00,False,PRODUIT_COMMERCIAL,REGLE_PAR_DEFAUT,False,NaN,True,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/10 8:26,2.75,17850.0,United Kingdom,2,4,...,2.75,False,2010-12-01 08:26:00,False,PRODUIT_COMMERCIAL,REGLE_PAR_DEFAUT,False,NaN,True,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/10 8:26,3.39,17850.0,United Kingdom,3,5,...,3.39,False,2010-12-01 08:26:00,False,PRODUIT_COMMERCIAL,REGLE_PAR_DEFAUT,False,NaN,True,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/10 8:26,3.39,17850.0,United Kingdom,4,6,...,3.39,False,2010-12-01 08:26:00,False,PRODUIT_COMMERCIAL,REGLE_PAR_DEFAUT,False,NaN,True,20.34



Copie de travail créée : df
Dimensions de df : (522364, 28)

===== RÉFÉRENCES DE LA SOURCE L2 =====
Nombre de lignes : 522364
Quantité totale : 5561117
Montant total : 10244661.18

===== TABLEAU RÉCAPITULATIF =====


,Indicateur,Valeur
0,Nombre de lignes,522364.00
1,Nombre de colonnes,28.00
2,Quantité totale,5561117.00
3,Montant total,10244661.18



VALIDATION : les huit colonnes métier sont présentes.

ÉTAPE 01 TERMINÉE.


# ÉTAPE 02 — Formuler et tester le grain
**Durée indicative : 15 min · Objectif :** Définir le sens d’une observation avant de créer le fait.

**Repère :** CP3 — faire valider une phrase précise et la confronter aux lignes.

## Questions du TP
1. Quelle phrase décrit exactement une ligne du fait ?
2. Une ligne de FactVente correspond-elle à une seule observation de vente au niveau défini ?
3. Que perd-on si le grain reste ambigu ?

## Question complémentaire — prendre du recul
Deux lignes ont le même InvoiceNo et le même StockCode, mais des quantités différentes. Faut-il les supprimer, les regrouper ou les conserver ? Justifiez selon le grain et les règles du TP2.

### Réponses du binôme — à compléter

**Notre phrase de grain :**  
Une ligne de la table FactVente représente une ligne de vente d’un produit sur une facture donnée. Elle contient les informations concernant le produit vendu, la quantité, le prix unitaire, la date de vente, le client et le pays. Chaque ligne représente une observation individuelle de vente.

| Exemple de ligne L2 — SourceIndex | Ce que représente cette ligne | Compatible avec le grain ? Pourquoi ? |
|---|---|---|
| Première ligne affichée dans la source L2 | Une ligne de vente d'un produit sur une facture, avec sa quantité et son prix unitaire. | Oui, car elle représente une observation individuelle de vente. |
| Deuxième ligne affichée dans la source L2 | Une autre ligne de vente contenant les informations d'un produit vendu. | Oui, car elle correspond au niveau de détail choisi. |
| Troisième ligne affichée dans la source L2 | Une ligne de transaction associée à une facture et à un produit. | Oui, car elle représente une observation de vente distincte. |

**Notre justification :**  
Nous avons choisi comme grain une ligne de vente individuelle. Ce choix permet de conserver le détail des transactions et d'analyser les ventes selon plusieurs critères, comme le produit, la date, le client et le pays. Chaque observation est conservée séparément afin de préserver les informations d'origine et de garantir la traçabilité grâce à SourceIndex.

**Point à vérifier / difficulté rencontrée :**  
Nous avons vérifié les répétitions du couple InvoiceNo + StockCode, distingué ces répétitions des doublons exacts et contrôlé l'unicité de SourceIndex. La principale difficulté consiste à différencier une répétition normale d'une véritable duplication de données. Aucune ligne n'a été supprimée ou regroupée pendant cette étape.

In [3]:
# ============================================================
# ÉTAPE 02 — FORMULER ET TESTER LE GRAIN
# Objectif : comprendre ce que représente une ligne de vente
# ============================================================

import pandas as pd
from IPython.display import display

# ------------------------------------------------------------
# 1. Vérifier les colonnes nécessaires
# ------------------------------------------------------------

colonnes_utiles = [
    "SourceIndex",
    "SourceRow",
    "SourceFile",
    "InvoiceNo",
    "StockCode",
    "Description",
    "Quantity",
    "UnitPrice",
    "InvoiceDate",
    "CustomerID",
    "Country"
]

colonnes_presentes = [
    col for col in colonnes_utiles
    if col in df_prep.columns
]

print("Colonnes disponibles pour l'analyse :")
print(colonnes_presentes)


# ------------------------------------------------------------
# 2. Afficher trois lignes L2 avec leur provenance
# ------------------------------------------------------------

print("\n1. EXEMPLES DE TROIS LIGNES DE LA SOURCE L2")

exemples_l2 = df_prep[colonnes_presentes].head(3)

display(exemples_l2)

print("Chaque ligne affichée représente une observation de la source L2.")


# ------------------------------------------------------------
# 3. Rechercher les répétitions InvoiceNo + StockCode
# ------------------------------------------------------------

print("\n2. RECHERCHE DES RÉPÉTITIONS InvoiceNo + StockCode")

# On compte le nombre de lignes pour chaque couple facture-produit.
repetitions = (
    df_prep
    .groupby(["InvoiceNo", "StockCode"], dropna=False)
    .size()
    .reset_index(name="Nombre_lignes")
)

# On conserve uniquement les couples présents plusieurs fois.
repetitions = repetitions[
    repetitions["Nombre_lignes"] > 1
].sort_values("Nombre_lignes", ascending=False)

print("Nombre de couples InvoiceNo + StockCode répétés :",
      len(repetitions))

print("\nQuelques exemples de couples répétés :")
display(repetitions.head(10))


# ------------------------------------------------------------
# 4. Afficher les lignes correspondant à quelques répétitions
# ------------------------------------------------------------

print("\n3. DÉTAIL DE QUELQUES RÉPÉTITIONS")

if len(repetitions) > 0:

    # Sélectionner les cinq premiers couples répétés.
    couples_exemples = repetitions[
        ["InvoiceNo", "StockCode"]
    ].head(5)

    # Récupérer les lignes L2 correspondantes.
    details_repetitions = df_prep.merge(
        couples_exemples,
        on=["InvoiceNo", "StockCode"],
        how="inner"
    )

    colonnes_details = [
        col for col in [
            "SourceIndex",
            "SourceRow",
            "InvoiceNo",
            "StockCode",
            "Description",
            "Quantity",
            "UnitPrice",
            "InvoiceDate"
        ]
        if col in details_repetitions.columns
    ]

    display(
        details_repetitions[colonnes_details]
        .sort_values(["InvoiceNo", "StockCode", "SourceIndex"])
    )

else:
    print("Aucune répétition InvoiceNo + StockCode trouvée.")


# ------------------------------------------------------------
# 5. Distinguer répétition du couple et doublon exact
# ------------------------------------------------------------

print("\n4. DISTINCTION ENTRE RÉPÉTITION ET DOUBLON EXACT")

# Les huit colonnes métier utilisées dans la règle de TP2.
colonnes_metier = [
    "InvoiceNo",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "UnitPrice",
    "CustomerID",
    "Country"
]

# Vérifier que les huit colonnes existent.
colonnes_metier_absentes = [
    col for col in colonnes_metier
    if col not in df_prep.columns
]

if colonnes_metier_absentes:
    raise ValueError(
        f"Colonnes métier manquantes : {colonnes_metier_absentes}"
    )

# Nombre de lignes appartenant à un groupe de doublons exacts.
doublons_exacts_lignes = df_prep.duplicated(
    subset=colonnes_metier,
    keep=False
).sum()

# Nombre de doublons exacts supplémentaires
# (en conservant la première occurrence).
doublons_exacts_supplementaires = df_prep.duplicated(
    subset=colonnes_metier,
    keep="first"
).sum()

print("Lignes appartenant à des groupes de doublons exacts :",
      doublons_exacts_lignes)

print("Doublons exacts supplémentaires :",
      doublons_exacts_supplementaires)

# Afficher quelques exemples de doublons exacts, s'il y en a.
if doublons_exacts_lignes > 0:
    exemples_doublons = df_prep[
        df_prep.duplicated(subset=colonnes_metier, keep=False)
    ]

    colonnes_affichage = [
        col for col in [
            "SourceIndex",
            "InvoiceNo",
            "StockCode",
            "Description",
            "Quantity",
            "UnitPrice"
        ]
        if col in exemples_doublons.columns
    ]

    print("\nExemples de doublons exacts selon les huit colonnes métier :")
    display(exemples_doublons[colonnes_affichage].head(10))

else:
    print("Aucun doublon exact selon les huit colonnes métier.")


# ------------------------------------------------------------
# 6. Vérifier l'unicité de SourceIndex
# ------------------------------------------------------------

print("\n5. CONTRÔLE DE L'UNICITÉ DE SourceIndex")

if "SourceIndex" in df_prep.columns:

    nombre_lignes = len(df_prep)
    nombre_sourceindex_distincts = df_prep["SourceIndex"].nunique(
        dropna=False
    )
    nombre_sourceindex_manquants = df_prep["SourceIndex"].isna().sum()
    nombre_sourceindex_repetes = df_prep["SourceIndex"].duplicated(
        keep=False
    ).sum()

    print("Nombre total de lignes :", nombre_lignes)
    print("Nombre de SourceIndex distincts :", nombre_sourceindex_distincts)
    print("Nombre de SourceIndex manquants :", nombre_sourceindex_manquants)
    print("Nombre de lignes avec un SourceIndex répété :",
          nombre_sourceindex_repetes)

    if (
        nombre_sourceindex_manquants == 0
        and nombre_sourceindex_repetes == 0
    ):
        print("Résultat : SourceIndex est unique et non manquant.")
    else:
        print("Résultat : SourceIndex n'est pas parfaitement unique.")

else:
    print("La colonne SourceIndex est absente de la source.")


# ------------------------------------------------------------
# 7. Vérifier que l'effectif n'a pas changé
# ------------------------------------------------------------

print("\n6. CONTRÔLE FINAL — AUCUNE SUPPRESSION")

effectif_avant = len(df_prep)
effectif_apres = len(df_prep)

print("Effectif avant l'analyse :", effectif_avant)
print("Effectif après l'analyse :", effectif_apres)

if effectif_avant == effectif_apres:
    print("VALIDATION : aucune ligne n'a été supprimée ou regroupée.")
else:
    print("ATTENTION : l'effectif a changé.")

print("\nÉTAPE 02 TERMINÉE.")

Colonnes disponibles pour l'analyse :
['SourceIndex', 'SourceRow', 'SourceFile', 'InvoiceNo', 'StockCode', 'Description', 'Quantity', 'UnitPrice', 'InvoiceDate', 'CustomerID', 'Country']

1. EXEMPLES DE TROIS LIGNES DE LA SOURCE L2


,SourceIndex,SourceRow,SourceFile,InvoiceNo,StockCode,Description,Quantity,UnitPrice,InvoiceDate,CustomerID,Country
0,0,2,Online_Retail.csv,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2.55,12/1/10 8:26,17850.0,United Kingdom
1,1,3,Online_Retail.csv,536365,71053,WHITE METAL LANTERN,6,3.39,12/1/10 8:26,17850.0,United Kingdom
2,2,4,Online_Retail.csv,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2.75,12/1/10 8:26,17850.0,United Kingdom


Chaque ligne affichée représente une observation de la source L2.

2. RECHERCHE DES RÉPÉTITIONS InvoiceNo + StockCode
Nombre de couples InvoiceNo + StockCode répétés : 5078

Quelques exemples de couples répétés :


,InvoiceNo,StockCode,Nombre_lignes
343946,568188,84879,4
279395,562549,20829,4
427643,574965,23084,4
56215,541266,21621,4
498439,580399,21109,4
378318,571046,22197,4
399499,572703,21818,4
455248,577034,22065,3
419916,574477,22578,3
331380,567184,22696,3



3. DÉTAIL DE QUELQUES RÉPÉTITIONS


,SourceIndex,SourceRow,InvoiceNo,StockCode,Description,Quantity,UnitPrice,InvoiceDate
0,58959,58961,541266,21621,VINTAGE UNION JACK BUNTING,12,8.50,1/16/11 16:25
1,58960,58962,541266,21621,VINTAGE UNION JACK BUNTING,13,8.50,1/16/11 16:25
2,58961,58963,541266,21621,VINTAGE UNION JACK BUNTING,10,8.50,1/16/11 16:25
3,58963,58965,541266,21621,VINTAGE UNION JACK BUNTING,7,8.50,1/16/11 16:25
4,292289,292291,562549,20829,GLITTER HANGING BUTTERFLY STRING,5,2.10,8/5/11 16:10
5,292293,292295,562549,20829,GLITTER HANGING BUTTERFLY STRING,2,2.10,8/5/11 16:10
6,292300,292302,562549,20829,GLITTER HANGING BUTTERFLY STRING,1,2.10,8/5/11 16:10
7,292304,292306,562549,20829,GLITTER HANGING BUTTERFLY STRING,3,2.10,8/5/11 16:10
8,359418,359420,568188,84879,ASSORTED COLOUR BIRD ORNAMENT,5,1.69,9/25/11 14:33
9,359421,359423,568188,84879,ASSORTED COLOUR BIRD ORNAMENT,2,1.69,9/25/11 14:33



4. DISTINCTION ENTRE RÉPÉTITION ET DOUBLON EXACT
Lignes appartenant à des groupes de doublons exacts : 0
Doublons exacts supplémentaires : 0
Aucun doublon exact selon les huit colonnes métier.

5. CONTRÔLE DE L'UNICITÉ DE SourceIndex
Nombre total de lignes : 522364
Nombre de SourceIndex distincts : 522364
Nombre de SourceIndex manquants : 0
Nombre de lignes avec un SourceIndex répété : 0
Résultat : SourceIndex est unique et non manquant.

6. CONTRÔLE FINAL — AUCUNE SUPPRESSION
Effectif avant l'analyse : 522364
Effectif après l'analyse : 522364
VALIDATION : aucune ligne n'a été supprimée ou regroupée.

ÉTAPE 02 TERMINÉE.


### Preuve de l’étape — à renseigner après exécution
- **Variables ou tables obtenues :**
  Nous avons obtenu les variables suivantes :
- exemples_l2 : contient trois exemples de lignes de la source L2 avec leurs informations métier et leur provenance.
- repetitions : contient les couples InvoiceNo + StockCode répétés. Nous avons identifié 5 078 couples répétés.
- details_repetitions : contient les détails des lignes correspondant aux couples répétés, notamment leurs quantités, leurs prix et leurs descriptions.
- doublons_exacts_lignes : nombre de lignes appartenant à des groupes de doublons exacts.
- controles : résultats des vérifications de l'unicité de SourceIndex et de la conservation de l'effectif.

- **Résultat du contrôle / observations :**

L'analyse a montré que plusieurs lignes possèdent le même couple InvoiceNo + StockCode. Par exemple, pour la facture 541266 et le produit 21621, nous avons trouvé quatre lignes avec des quantités différentes : 12, 13, 10 et 7. Nous avons également constaté que ces lignes peuvent avoir la même description, le même prix unitaire et la même date, mais des quantités différentes.

Le contrôle des doublons exacts selon les huit colonnes métier a donné 0 doublon exact. De plus, SourceIndex est unique et ne contient aucune valeur manquante : les 522 364 lignes possèdent 522 364 valeurs distinctes de SourceIndex.

Enfin, l'effectif de la source est resté inchangé : 522 364 lignes avant et après l'analyse.

- **Décision ou correction :** 
Nous avons retenu comme grain une ligne de vente individuelle correspondant à une observation de la source L2. Les répétitions du couple InvoiceNo + StockCode sont conservées, car elles ne représentent pas nécessairement des doublons. Les lignes ayant des quantités différentes doivent rester séparées afin de préserver le détail des transactions. Aucun doublon exact n'ayant été détecté, aucune suppression ni correction n'a été nécessaire. La source df_prep reste inchangée et SourceIndex permet d'assurer la traçabilité de chaque observation.

# ÉTAPE 03 — Choisir les mesures au bon grain
**Durée indicative : 15 min · Objectif :** Distinguer mesure, identifiant et attribut descriptif.



## Questions du TP
1. Quelles données sont des mesures et lesquelles sont des axes d’analyse ?
2. Quels contrôles prouveront que le fait respecte son grain ?

## Question complémentaire — prendre du recul
Peut-on répéter le montant total de la commande sur chaque ligne et l’additionner ? Le prix unitaire se somme-t-il comme un montant de vente ? Expliquez le risque dans les deux cas.

### Réponses du binôme — à compléter

| Colonne L2 | Rôle | Nom dans le modèle | Description |
|---|---|---|---|
| `Quantity` | Mesure | `QuantiteVendue` | Quantité vendue |
| `UnitPrice` | Mesure | `PrixUnitaire` | Prix d'une unité |
| `MontantLigne` | Mesure | `MontantLigne` | Montant total de la ligne |
| `InvoiceNo` | Identifiant | `InvoiceNo` | Numéro de facture |
| `StockCode` | Identifiant | `StockCode` | Code du produit |
| `Country` | descriptif | `Pays` | Pays du client |
| `CustomerID` | Identifiant | `CustomerID` | Identifiant du client |
| `InvoiceDate` | descriptif | `DateVente` | Date de la vente |

**Formule de MontantLigne et cohérence au grain :** La formule utilisée est :
\[
MontantLigne = Quantity \times UnitPrice
\]

Cette formule permet de calculer le montant de chaque ligne de vente. Elle est cohérente avec notre grain, car chaque observation représente une ligne de vente individuelle.

**Notre justification :**  
Nous avons choisi trois mesures principales : Quantity, UnitPrice et MontantLigne. Elles permettent d'analyser les quantités vendues, les prix unitaires et les montants des ventes. Chaque mesure est conservée au niveau de la ligne de vente afin de respecter le grain défini dans l'étape 2.

**Point à vérifier / difficulté rencontrée :**  
Nous devons vérifier les éventuels écarts entre les montants calculés et les montants fournis par TP2, ainsi que les valeurs manquantes. Nous devons également éviter de sommer directement les prix unitaires.

In [4]:

# ============================================================
# ÉTAPE 03 — CHOISIR LES MESURES AU BON GRAIN
# ============================================================

import pandas as pd
import numpy as np
from IPython.display import display

# 1. Identifier les colonnes numériques et leurs types

colonnes_numeriques = [
    "Quantity", "UnitPrice",
    "Quantity_num", "UnitPrice_num",
    "MontantLigne"
]

print("1. TYPES DES COLONNES NUMÉRIQUES")

for col in colonnes_numeriques:
    if col in df_prep.columns:
        print(col, ":", df_prep[col].dtype)

# 2. Créer une copie de travail sans modifier df_prep

df = df_prep.copy()

# Utiliser les colonnes numériques dérivées du TP2
# lorsqu'elles sont disponibles.

if "Quantity_num" in df.columns:
    quantite = pd.to_numeric(df["Quantity_num"], errors="coerce")
else:
    quantite = pd.to_numeric(df["Quantity"], errors="coerce")

if "UnitPrice_num" in df.columns:
    prix = pd.to_numeric(df["UnitPrice_num"], errors="coerce")
else:
    prix = pd.to_numeric(df["UnitPrice"], errors="coerce")

# 3. Calculer le montant théorique de chaque ligne

montant_calcule = quantite * prix

montant_source = pd.to_numeric(
    df["MontantLigne"], errors="coerce"
)

# Comparer le montant calculé au montant fourni par TP2

comparaison = pd.DataFrame({
    "Quantity": quantite,
    "UnitPrice": prix,
    "MontantLigne_source": montant_source,
    "MontantLigne_calcule": montant_calcule
})

# Identifier les lignes comparables

comparaison["Ecart"] = (
    comparaison["MontantLigne_source"]
    - comparaison["MontantLigne_calcule"]
)

comparaison["Montants_coherents"] = np.isclose(
    comparaison["MontantLigne_source"],
    comparaison["MontantLigne_calcule"],
    rtol=1e-9,
    atol=1e-8,
    equal_nan=False
)

# 4. Afficher quelques exemples

print("\n2. COMPARAISON DES MONTANTS DE LIGNE")

display(comparaison.head(10))

# 5. Afficher les résultats du contrôle

lignes_comparables = (
    comparaison["MontantLigne_source"].notna()
    & comparaison["MontantLigne_calcule"].notna()
)

nb_comparables = lignes_comparables.sum()

nb_coherents = (
    comparaison.loc[
        lignes_comparables, "Montants_coherents"
    ].sum()
)

nb_ecarts = nb_comparables - nb_coherents

print("\n3. RÉSULTATS DU CONTRÔLE")

print("Nombre de lignes comparables :", nb_comparables)
print("Nombre de montants cohérents :", nb_coherents)
print("Nombre de montants différents :", nb_ecarts)

print("Nombre de lignes non comparables :",
      len(df) - nb_comparables)

# Afficher les éventuels écarts

print("\n4. LIGNES PRÉSENTANT DES ÉCARTS")

ecarts = comparaison.loc[
    lignes_comparables
    & (~comparaison["Montants_coherents"])
]

display(ecarts.head(10))

# 6. Calculer les totaux de référence

print("\n5. TOTAUX DE RÉFÉRENCE L2")

print("Nombre total de lignes :", len(df))

print("Quantité totale :",
      quantite.sum())

print("Montant total :",
      montant_source.sum())

print("Montant recalculé :",
      montant_calcule.sum())

# 7. Documenter le mapping des mesures

mapping_mesures = pd.DataFrame({
    "Colonne_L2": [
        "Quantity",
        "UnitPrice",
        "MontantLigne"
    ],
    "Colonne_modele": [
        "QuantiteVendue",
        "PrixUnitaire",
        "MontantLigne"
    ],
    "Role": [
        "Quantité vendue",
        "Prix d'une unité",
        "Montant de la ligne"
    ]
})

print("\n6. MAPPING DES MESURES")

display(mapping_mesures)

print("\nÉTAPE 03 TERMINÉE.")


1. TYPES DES COLONNES NUMÉRIQUES
Quantity : int64
UnitPrice : float64
Quantity_num : int64
UnitPrice_num : float64
MontantLigne : float64

2. COMPARAISON DES MONTANTS DE LIGNE


,Quantity,UnitPrice,MontantLigne_source,MontantLigne_calcule,Ecart,Montants_coherents
0,6,2.55,15.30,15.30,1.776357e-15,True
1,6,3.39,20.34,20.34,0.000000e+00,True
2,8,2.75,22.00,22.00,0.000000e+00,True
3,6,3.39,20.34,20.34,0.000000e+00,True
4,6,3.39,20.34,20.34,0.000000e+00,True
5,2,7.65,15.30,15.30,0.000000e+00,True
6,6,4.25,25.50,25.50,0.000000e+00,True
7,6,1.85,11.10,11.10,-1.776357e-15,True
8,6,1.85,11.10,11.10,-1.776357e-15,True
9,32,1.69,54.08,54.08,0.000000e+00,True



3. RÉSULTATS DU CONTRÔLE
Nombre de lignes comparables : 522364
Nombre de montants cohérents : 522364
Nombre de montants différents : 0
Nombre de lignes non comparables : 0

4. LIGNES PRÉSENTANT DES ÉCARTS


,Quantity,UnitPrice,MontantLigne_source,MontantLigne_calcule,Ecart,Montants_coherents



5. TOTAUX DE RÉFÉRENCE L2
Nombre total de lignes : 522364
Quantité totale : 5561117
Montant total : 10244661.18
Montant recalculé : 10244661.18

6. MAPPING DES MESURES


,Colonne_L2,Colonne_modele,Role
0,Quantity,QuantiteVendue,Quantité vendue
1,UnitPrice,PrixUnitaire,Prix d'une unité
2,MontantLigne,MontantLigne,Montant de la ligne



ÉTAPE 03 TERMINÉE.


### Preuve de l’étape — à renseigner après exécution
- **Variables ou tables obtenues :** Nous avons obtenu les variables suivantes :
- df : copie de travail de la source L2, utilisée pour les analyses sans modifier df_prep.
- quantite : contient les quantités vendues à partir de Quantity_num.
- prix : contient les prix unitaires à partir de UnitPrice_num.
- montant_calcule : contient les montants calculés avec la formule Quantity × UnitPrice.
- montant_source : contient les montants fournis par L2 dans MontantLigne.
- comparaison : tableau permettant de comparer les montants calculés avec ceux de la source.
- ecarts : contient les éventuelles lignes présentant des différences de montants.
- mapping_mesures : tableau présentant la correspondance entre les colonnes L2 et les mesures du modèle.

- **Résultat du contrôle / observations :** La comparaison a montré que les *522 364 lignes* possèdent des montants cohérents entre les valeurs calculées et celles fournies par L2.
- Nombre de lignes comparables : 522 364.
- Nombre de montants cohérents : 522 364.
- Nombre de montants différents : 0.
- Nombre de lignes non comparables : 0.
Nous avons également obtenu les mêmes totaux pour les montants :
- Quantité totale : 5 561 117.
- Montant total L2 : 10 244 661,18.
- Montant total recalculé : 10 244 661,18.
Ces résultats confirment que la formule MontantLigne = Quantity × UnitPrice est respectée sur toutes les lignes.

- **Décision ou correction :** Nous avons décidé de conserver les trois mesures principales dans le modèle décisionnel : QuantiteVendue, PrixUnitaire et MontantLigne.
Le mapping retenu est :
- Quantity → QuantiteVendue.
- UnitPrice → PrixUnitaire.
- MontantLigne → MontantLigne.
Aucune correction n'a été nécessaire, car tous les montants sont cohérents et aucune différence n'a été détectée. Nous allons donc conserver les valeurs de L2 et utiliser MontantLigne pour analyser le chiffre d'affaires, sans additionner directement les prix unitaires.

# ÉTAPE 04 — Concevoir le modèle avant l’alimentation
**Durée indicative : 20 min · Objectif :** Définir les tables et leurs relations.

**Repère :** Préparation de CP4 — expliquer chaque table, pas seulement dessiner une étoile.

## Questions du TP
1. Pourquoi InvoiceNo reste-t-il une dimension dégénérée ?
2. Pourquoi DimPays est-elle séparée de DimClient ?
3. Pourquoi les dimensions doivent-elles être construites avant le rattachement du fait ?

## Question complémentaire — prendre du recul
Distinguez le Data Warehouse minimal construit, le Data Mart Ventes destiné à un domaine et un simple export CSV. Le fait de disposer de cinq fichiers prouve-t-il, à lui seul, qu’un entrepôt exploitable est construit ?

### Réponses du binôme — à compléter

| Table | Une ligne représente… | Clé primaire | Références / attributs / mesures |
|---|---|---|---|
| FactVente | Une ligne de vente d’un produit sur une facture | FactVenteKey | DateKey, ProduitKey, ClientKey, PaysKey, InvoiceNo, Quantity, UnitPrice, MontantLigne, SourceIndex |
| DimDate | Un jour du calendrier | DateKey | Date, Jour, Mois, Année, Trimestre |
| DimProduit | Un produit identifié par son code | ProduitKey | StockCode_norm, Description_norm |
| DimClient | Un client identifié par son numéro | ClientKey | ClientID_norm |
| DimPays | Un pays | PaysKey | Country_norm |

**Schéma proposé :** Modèle en étoile
Le modèle proposé est un schéma en étoile composé d'une table de faits `FactVente` et de quatre tables de dimensions : `DimDate`, `DimProduit`, `DimClient` et `DimPays`.

*1. Table de faits : FactVente*
- Clé primaire : `FactVenteKey`
- Clés étrangères : `DateKey`, `ProduitKey`, `ClientKey`, `PaysKey`
- Mesures : `QuantiteVendue`, `PrixUnitaire`, `MontantLigne`
- Autres attributs : `InvoiceNo`, `SourceIndex`

*2. Tables de dimensions :*
- `DimDate` : Date, Jour, Mois, Année, Trimestre.
- `DimProduit` : StockCode_norm, Description_norm.
- `DimClient` : CustomerID_norm.
- `DimPays` : Country_norm.

**3. Relations entre les tables :**

Chaque dimension est liée à la table `FactVente` par une relation de type **1 à N (un-à-plusieurs)**. Une dimension peut être associée à plusieurs lignes de vente.
Ce modèle facilite l'analyse des ventes selon la date, le produit, le client et le pays.

**Cardinalités et sens de chaque relation :** 
DimDate (1) → FactVente (N) : un jour peut contenir plusieurs ventes.
DimProduit (1) → FactVente (N) : un produit peut apparaître dans plusieurs ventes.
DimClient (1) → FactVente (N) : un client peut effectuer plusieurs achats.
DimPays (1) → FactVente (N) : un pays peut être associé à plusieurs ventes.

**Notre justification :**  
Nous avons choisi un modèle en étoile composé d'une table de faits FactVente et de quatre dimensions : DimDate, DimProduit, DimClient et DimPays.
La table FactVente contient les mesures des ventes et les clés étrangères permettant de relier les différentes dimensions. Chaque dimension représente un axe d'analyse spécifique.
Ce modèle facilite l'analyse des ventes selon le temps, les produits, les clients et les pays, tout en conservant le grain initial des données.

**Point à vérifier / difficulté rencontrée :**  
La principale difficulté consiste à définir correctement les clés primaires et étrangères pour garantir la cohérence des relations entre les tables.
Nous devons également prévoir un client inconnu pour conserver les ventes dont le CustomerID est absent et vérifier que chaque vente possède des références valides vers les dimensions.

In [5]:
import pandas as pd

# Description des tables du modèle
modele = [
    {
        "Table": "FactVente",
        "Grain": "Une ligne de vente",
        "Cle_primaire": "FactVenteKey",
        "Cle_de_rattachement": "DateKey, ProduitKey, ClientKey, PaysKey",
        "Attributs_mesures": "InvoiceNo, Quantity, UnitPrice, MontantLigne, SourceIndex"
    },
    {
        "Table": "DimDate",
        "Grain": "Un jour",
        "Cle_primaire": "DateKey",
        "Cle_de_rattachement": "InvoiceDate_dt",
        "Attributs_mesures": "Date, Jour, Mois, Année, Trimestre"
    },
    {
        "Table": "DimProduit",
        "Grain": "Un produit",
        "Cle_primaire": "ProduitKey",
        "Cle_de_rattachement": "StockCode_norm",
        "Attributs_mesures": "Description_norm"
    },
    {
        "Table": "DimClient",
        "Grain": "Un client",
        "Cle_primaire": "ClientKey",
        "Cle_de_rattachement": "CustomerID_norm",
        "Attributs_mesures": "CustomerID_norm"
    },
    {
        "Table": "DimPays",
        "Grain": "Un pays",
        "Cle_primaire": "PaysKey",
        "Cle_de_rattachement": "Country_norm",
        "Attributs_mesures": "Country_norm"
    }
]

df_modele = pd.DataFrame(modele)

display(df_modele)

,Table,Grain,Cle_primaire,Cle_de_rattachement,Attributs_mesures
0,FactVente,Une ligne de vente,FactVenteKey,"DateKey, ProduitKey, ClientKey, PaysKey","InvoiceNo, Quantity, UnitPrice, MontantLigne, ..."
1,DimDate,Un jour,DateKey,InvoiceDate_dt,"Date, Jour, Mois, Année, Trimestre"
2,DimProduit,Un produit,ProduitKey,StockCode_norm,Description_norm
3,DimClient,Un client,ClientKey,CustomerID_norm,CustomerID_norm
4,DimPays,Un pays,PaysKey,Country_norm,Country_norm


### Preuve de l’étape — à renseigner après exécution
- **Variables ou tables obtenues :** Nous avons obtenu la variable df_modele, un DataFrame descriptif présentant les cinq tables du modèle décisionnel, leur grain, leurs clés primaires, leurs clés de rattachement et leurs attributs ou mesures.
 
- **Résultat du contrôle / observations :** Le tableau affiche les cinq tables prévues : FactVente, DimDate, DimProduit, DimClient et DimPays. Chaque table possède un grain et une clé primaire définis. Les clés de rattachement nécessaires à la construction des relations sont également identifiées.
 
- **Décision ou correction :** Nous avons retenu un modèle en étoile composé d'une table de faits et de quatre dimensions. Les dimensions seront construites en premier afin de générer les clés nécessaires au rattachement des ventes. Aucune modification des données sources n'a été effectuée pendant cette étape.

# ÉTAPE 05 — Construire DimDate au grain du jour
**Durée indicative : 20 min · Objectif :** Créer des références temporelles uniques et des niveaux cohérents.



## Questions du TP
1. Construire DimDate à partir des données préparées et documenter la règle de clé.
2. Comment vérifier l’unicité des clés et la couverture des valeurs utilisées par le fait ?

## Question complémentaire — prendre du recul
Le code de départ de la fiche extrait les horodatages distincts avant de créer une clé au jour. Que se passe-t-il pour deux ventes le même jour à 8 h et 15 h ? Comment corriger l’ordre des opérations sans perdre l’heure de la vente ?

### Réponses du binôme — à compléter

**Colonne de date choisie et format vérifié :** La colonne choisie est InvoiceDate_dt. Elle contient la date et l’heure de chaque vente. Nous vérifions que son format est de type datetime. 
**Grain de DimDate :** Une ligne représente un seul jour du calendrier, sans tenir compte de l’heure. 
**Règle de DateKey :** La clé est construite au format AAAAMMJJ. Par exemple, le 1er décembre 2010 donne la clé 20101201.

| Horodatage d’une vente | Jour de rattachement | DateKey attendue selon notre règle |
|---|---|---|
| 2010-12-01 08:26:00 | 2010-12-01 | 20101201 |
| 2010-12-01 15:00:00 | 2010-12-01 | 20101201 |

**Hiérarchie temporelle proposée :** Année → Trimestre → Mois → Jour.  
**Où l’horodatage reste-t-il conservé ?** L’horodatage complet reste conservé dans la table FactVente, dans la colonne InvoiceDate_dt. Dans DimDate, nous conservons uniquement le jour.

**Notre justification :**  
Nous utilisons une seule ligne dans DimDate pour chaque jour. Ainsi, deux ventes effectuées le même jour, même à des heures différentes, utilisent la même DateKey. Cela évite de créer plusieurs lignes pour une même journée.

**Point à vérifier / difficulté rencontrée :**  
Nous devons vérifier le format des dates, les éventuelles conversions impossibles et l'unicité de DateKey.

In [6]:
import pandas as pd

# ============================================================
# 1. Examiner les dates fournies
# ============================================================

print("1. EXAMEN DES DATES")

print("Type de la colonne :", df["InvoiceDate_dt"].dtype)
print("Exemples de dates :")
display(df[["InvoiceDate_dt"]].head())

# ============================================================
# 2. Convertir les dates en conservant l'heure
# ============================================================

df["InvoiceDate_horodatage"] = pd.to_datetime(
    df["InvoiceDate_dt"],
    errors="coerce"
)

# Vérifier les conversions impossibles
nb_dates_invalides = df["InvoiceDate_horodatage"].isna().sum()

print("\n2. CONTRÔLE DES CONVERSIONS")
print("Nombre de dates invalides :", nb_dates_invalides)

# ============================================================
# 3. Dériver une date au grain du jour
#    L'heure reste conservée dans InvoiceDate_horodatage
# ============================================================

df["Date_vente"] = df["InvoiceDate_horodatage"].dt.normalize()

# ============================================================
# 4. Extraire les jours distincts
# ============================================================

jours_distincts = (
    df["Date_vente"]
    .dropna()
    .drop_duplicates()
    .sort_values()
)

# ============================================================
# 5. Construire la dimension DimDate
# ============================================================

dim_date = pd.DataFrame({"Date": jours_distincts})

# Clé au format AAAAMMJJ
dim_date["DateKey"] = (
    dim_date["Date"].dt.strftime("%Y%m%d").astype("int64")
)

# Attributs temporels
dim_date["Jour"] = dim_date["Date"].dt.day
dim_date["NumMois"] = dim_date["Date"].dt.month

noms_mois = {
    1: "Janvier", 2: "Février", 3: "Mars",
    4: "Avril", 5: "Mai", 6: "Juin",
    7: "Juillet", 8: "Août", 9: "Septembre",
    10: "Octobre", 11: "Novembre", 12: "Décembre"
}

dim_date["Mois"] = dim_date["NumMois"].map(noms_mois)
dim_date["Trimestre"] = "T" + dim_date["Date"].dt.quarter.astype(str)
dim_date["Annee"] = dim_date["Date"].dt.year
dim_date["AnneeMois"] = dim_date["Date"].dt.strftime("%Y-%m")

# Organiser les colonnes
dim_date = dim_date[
    ["DateKey", "Date", "Jour", "NumMois",
     "Mois", "Trimestre", "Annee", "AnneeMois"]
].reset_index(drop=True)

# ============================================================
# 6. Vérifier l'unicité des clés
# ============================================================

print("\n3. CONTRÔLE DE DIMDATE")

print("Nombre de lignes :", len(dim_date))
print("Nombre de clés uniques :", dim_date["DateKey"].nunique())
print("Présence de clés nulles :", dim_date["DateKey"].isna().sum())
print("Clés uniques :", dim_date["DateKey"].is_unique)

# Afficher un extrait
display(dim_date.head(10))

# ============================================================
# 7. Rattacher les ventes à DimDate
#    On conserve l'horodatage complet dans la table des ventes
# ============================================================

correspondance_dates = dict(
    zip(dim_date["Date"], dim_date["DateKey"])
)

df["DateKey"] = (
    df["Date_vente"]
    .map(correspondance_dates)
    .astype("Int64")
)

# ============================================================
# 8. Vérifier la couverture des dates utilisées par les ventes
# ============================================================

ventes_avec_date = df["Date_vente"].notna()

nb_ventes_valides = ventes_avec_date.sum()
nb_ventes_rattachees = df.loc[ventes_avec_date, "DateKey"].notna().sum()
nb_ventes_non_rattachees = (
    df.loc[ventes_avec_date, "DateKey"].isna().sum()
)

print("\n4. CONTRÔLE DE COUVERTURE")

print("Ventes avec une date valide :", nb_ventes_valides)
print("Ventes rattachées à DimDate :", nb_ventes_rattachees)
print("Ventes non rattachées :", nb_ventes_non_rattachees)
print("Ventes avec date invalide :", nb_dates_invalides)

# ============================================================
# 9. Vérifier le cas de deux ventes le même jour
# ============================================================

exemple = pd.DataFrame({
    "Horodatage": pd.to_datetime([
        "2010-12-01 08:00:00",
        "2010-12-01 15:00:00"
    ])
})

exemple["Jour_rattachement"] = exemple["Horodatage"].dt.normalize()
exemple["DateKey_attendue"] = (
    exemple["Jour_rattachement"].dt.strftime("%Y%m%d").astype("int64")
)

print("\n5. EXEMPLE DE RATTACHEMENT")
display(exemple)

# ============================================================
# 10. Résumé de l'étape
# ============================================================

print("\n6. PREUVE DE L'ÉTAPE 05")
print("Variable obtenue : dim_date")
print("Grain : un jour")
print("Règle de DateKey : AAAAMMJJ")
print("Hiérarchie : Année > Trimestre > Mois > Jour")
print("Horodatage complet conservé dans : InvoiceDate_horodatage")

1. EXAMEN DES DATES
Type de la colonne : object
Exemples de dates :


,InvoiceDate_dt
0,2010-12-01 08:26:00
1,2010-12-01 08:26:00
2,2010-12-01 08:26:00
3,2010-12-01 08:26:00
4,2010-12-01 08:26:00



2. CONTRÔLE DES CONVERSIONS
Nombre de dates invalides : 0

3. CONTRÔLE DE DIMDATE
Nombre de lignes : 305
Nombre de clés uniques : 305
Présence de clés nulles : 0
Clés uniques : True


,DateKey,Date,Jour,NumMois,Mois,Trimestre,Annee,AnneeMois
0,20101201,2010-12-01,1,12,Décembre,T4,2010,2010-12
1,20101202,2010-12-02,2,12,Décembre,T4,2010,2010-12
2,20101203,2010-12-03,3,12,Décembre,T4,2010,2010-12
3,20101205,2010-12-05,5,12,Décembre,T4,2010,2010-12
4,20101206,2010-12-06,6,12,Décembre,T4,2010,2010-12
5,20101207,2010-12-07,7,12,Décembre,T4,2010,2010-12
6,20101208,2010-12-08,8,12,Décembre,T4,2010,2010-12
7,20101209,2010-12-09,9,12,Décembre,T4,2010,2010-12
8,20101210,2010-12-10,10,12,Décembre,T4,2010,2010-12
9,20101212,2010-12-12,12,12,Décembre,T4,2010,2010-12



4. CONTRÔLE DE COUVERTURE
Ventes avec une date valide : 522364
Ventes rattachées à DimDate : 522364
Ventes non rattachées : 0
Ventes avec date invalide : 0

5. EXEMPLE DE RATTACHEMENT


,Horodatage,Jour_rattachement,DateKey_attendue
0,2010-12-01 08:00:00,2010-12-01,20101201
1,2010-12-01 15:00:00,2010-12-01,20101201



6. PREUVE DE L'ÉTAPE 05
Variable obtenue : dim_date
Grain : un jour
Règle de DateKey : AAAAMMJJ
Hiérarchie : Année > Trimestre > Mois > Jour
Horodatage complet conservé dans : InvoiceDate_horodatage


### Preuve de l’étape — à renseigner après exécution
- **Variables ou tables obtenues :**
  - dim_date : dimension temporelle contenant 305 jours distincts.
  - InvoiceDate_horodatage : conserve la date et l'heure complètes des ventes.
  - Date_vente : contient la date au grain du jour.
  - DateKey : clé de rattachement au format AAAAMMJJ.
- **Résultat du contrôle / observations :**
  - 305 lignes dans dim_date.
  - 305 clés uniques et aucune clé nulle.
  - Aucune date invalide détectée.
  - 522 364 ventes avec une date valide.
  - 522 364 ventes correctement rattachées à DimDate.
  - 0 vente non rattachée.
- **Décision ou correction :**
Nous avons conservé l'horodatage complet avant de réduire les dates au grain du jour. Nous avons ensuite extrait les jours distincts et créé DateKey au format AAAAMMJJ. Cette méthode garantit une seule ligne par jour dans DimDate, tout en conservant l'heure de chaque vente dans la table des faits.

# ÉTAPE 06 — Construire DimProduit sans inventer d’attributs
**Durée indicative : 20 min · Objectif :** Extraire les membres produit sans supprimer de ventes.



## Questions du TP
1. Construire DimProduit à partir de StockCode et Description.
2. Vérifier l’unicité des clés et la couverture des produits utilisés par le fait.

## Question complémentaire — prendre du recul
Quelle différence entre dédoublonner les ventes en TP2 et extraire les valeurs distinctes pour créer une dimension ? Si un même StockCode possède deux descriptions normalisées différentes, pourquoi une jointure sur le code seul peut-elle être ambiguë ?

### Réponses du binôme — à compléter

**Grain de DimProduit et colonnes de rapprochement :** Un produit est identifié par le couple StockCode_norm et Description_norm.  
**Traitement documenté d’une description absente, si ce cas existe :** Si une description est absente, elle est remplacée par la valeur technique DESCRIPTION_ABSENTE.  
**Règle de ProduitKey :** Une clé entière est attribuée à chaque couple distinct, après un tri déterministe.

**Pourquoi aucune catégorie ou marque n’est-elle ajoutée ?** Parce que ces informations ne sont pas disponibles dans les données préparées. Nous évitons donc d'inventer des attributs.  
**Les clés resteraient-elles stables si l’on ajoute de nouvelles valeurs et renumérote tout ?** Non. Une renumérotation complète peut modifier les clés déjà attribuées. Dans un système réel, il faut conserver les clés existantes et attribuer de nouvelles clés uniquement aux nouveaux produits.

**Notre justification :**  
Nous avons construit DimProduit à partir des couples distincts de codes et de descriptions normalisées. Cette méthode permet d'identifier les produits sans supprimer les lignes de ventes et de garantir un rapprochement plus précis avec la table de faits.

**Point à vérifier / difficulté rencontrée :**  
Vérifier l'unicité de ProduitKey, la couverture des couples utilisés par les ventes et les éventuels codes associés à plusieurs descriptions.

In [7]:
import pandas as pd
# ============================================================
# 1. EXAMINER LES COUPLES STOCKCODE / DESCRIPTION
# ============================================================

print("1. EXAMEN DES DONNÉES PRODUIT")

print("Nombre de lignes dans df :", len(df))

print("Colonnes disponibles :")
print(df[["StockCode_norm", "Description_norm"]].head())

# Créer une copie temporaire sans modifier df
produits_source = df[
    ["StockCode_norm", "Description_norm"]
].copy()

# Normalisation légère et traçable
produits_source["StockCode_norm"] = (
    produits_source["StockCode_norm"]
    .astype("string")
    .str.strip()
)

produits_source["Description_norm"] = (
    produits_source["Description_norm"]
    .astype("string")
    .str.strip()
)

# Traiter les valeurs absentes avec des valeurs techniques
nb_codes_absents = produits_source["StockCode_norm"].isna().sum()
nb_descriptions_absentes = produits_source["Description_norm"].isna().sum()

produits_source["StockCode_norm"] = (
    produits_source["StockCode_norm"]
    .fillna("CODE_ABSENT")
)

produits_source["Description_norm"] = (
    produits_source["Description_norm"]
    .fillna("DESCRIPTION_ABSENTE")
)

print("\nCodes absents :", nb_codes_absents)
print("Descriptions absentes :", nb_descriptions_absentes)


# ============================================================
# 2. EXTRAIRE LES COUPLES DISTINCTS
# ============================================================

produits_distincts = (
    produits_source[
        ["StockCode_norm", "Description_norm"]
    ]
    .drop_duplicates()
    .sort_values(
        ["StockCode_norm", "Description_norm"]
    )
    .reset_index(drop=True)
)

print("\n2. EXTRACTION DES PRODUITS")
print("Nombre de produits distincts :", len(produits_distincts))


# ============================================================
# 3. CONSTRUIRE DIM_PRODUIT ET ATTRIBUER PRODUITKEY
# ============================================================

dim_produit = produits_distincts.copy()

# Clé entière attribuée selon un ordre déterministe
dim_produit.insert(
    0,
    "ProduitKey",
    range(1, len(dim_produit) + 1)
)

print("\n3. EXTRAIT DE DIM_PRODUIT")
display(dim_produit.head(10))


# ============================================================
# 4. VÉRIFIER L'UNICITÉ DES CLÉS ET DU COUPLE
# ============================================================

print("\n4. CONTRÔLE D'UNICITÉ")

print("Nombre de lignes :", len(dim_produit))
print("Nombre de ProduitKey uniques :", dim_produit["ProduitKey"].nunique())
print("ProduitKey uniques :", dim_produit["ProduitKey"].is_unique)

print(
    "Couples StockCode / Description uniques :",
    not dim_produit.duplicated(
        ["StockCode_norm", "Description_norm"]
    ).any()
)

print(
    "Clés nulles :",
    dim_produit["ProduitKey"].isna().sum()
)


# ============================================================
# 5. EXAMINER LES CODES AVEC PLUSIEURS DESCRIPTIONS
# ============================================================

print("\n5. RECHERCHE DES CODES AMBIGUS")

nb_descriptions_par_code = (
    dim_produit.groupby("StockCode_norm")["Description_norm"]
    .nunique()
)

codes_ambigus = nb_descriptions_par_code[
    nb_descriptions_par_code > 1
]

print("Nombre de codes associés à plusieurs descriptions :",
      len(codes_ambigus))

if len(codes_ambigus) > 0:
    print("\nExemples de codes ambigus :")
    ambiguities = dim_produit[
        dim_produit["StockCode_norm"].isin(codes_ambigus.index)
    ]
    display(ambiguities.head(20))
else:
    print("Aucun code ambigu détecté.")


# ============================================================
# 6. VÉRIFIER LA COUVERTURE DES PRODUITS UTILISÉS PAR LE FAIT
# ============================================================

# Construire les couples présents dans les ventes
couples_ventes = produits_source[
    ["StockCode_norm", "Description_norm"]
].drop_duplicates()

# Comparer avec les couples de DimProduit
controle_couverture = couples_ventes.merge(
    dim_produit[
        ["StockCode_norm", "Description_norm"]
    ],
    on=["StockCode_norm", "Description_norm"],
    how="left",
    indicator=True
)

nb_couples_utilises = len(couples_ventes)

nb_couples_rattaches = (
    controle_couverture["_merge"] == "both"
).sum()

nb_couples_non_rattaches = (
    controle_couverture["_merge"] == "left_only"
).sum()

print("\n6. CONTRÔLE DE COUVERTURE")

print("Couples distincts utilisés par les ventes :", nb_couples_utilises)
print("Couples rattachés à DimProduit :", nb_couples_rattaches)
print("Couples non rattachés :", nb_couples_non_rattaches)


# ============================================================
# 7. AFFICHER LA PREUVE DE L'ÉTAPE 06
# ============================================================

print("\n7. PREUVE DE L'ÉTAPE 06")

print("Variable obtenue : dim_produit")
print("Grain : un couple StockCode / Description normalisée")
print("Clé primaire : ProduitKey")
print("Clé de rapprochement : StockCode_norm + Description_norm")

print("\nRÉSULTAT DES CONTRÔLES")
print("ProduitKey unique :", dim_produit["ProduitKey"].is_unique)
print("Couple unique :", not dim_produit.duplicated(
    ["StockCode_norm", "Description_norm"]
).any())
print("Couverture complète :", nb_couples_non_rattaches == 0)

print("\nDÉCISION")
print("Aucune vente n'a été supprimée.")
print("Aucune catégorie ou marque n'a été inventée.")
print("La dimension est construite à partir des valeurs distinctes.")

1. EXAMEN DES DONNÉES PRODUIT
Nombre de lignes dans df : 522364
Colonnes disponibles :
  StockCode_norm                     Description_norm
0         85123A   WHITE HANGING HEART T-LIGHT HOLDER
1          71053                  WHITE METAL LANTERN
2         84406B       CREAM CUPID HEARTS COAT HANGER
3         84029G  KNITTED UNION FLAG HOT WATER BOTTLE
4         84029E       RED WOOLLY HOTTIE WHITE HEART.

Codes absents : 0
Descriptions absentes : 0

2. EXTRACTION DES PRODUITS
Nombre de produits distincts : 4026

3. EXTRAIT DE DIM_PRODUIT


,ProduitKey,StockCode_norm,Description_norm
0,1,10002,INFLATABLE POLITICAL GLOBE
1,2,10080,GROOVY CACTUS INFLATABLE
2,3,10120,DOGGY RUBBER
3,4,10123C,HEARTS WRAPPING TAPE
4,5,10124A,SPOTS ON RED BOOKCOVER TAPE
5,6,10124G,ARMY CAMO BOOKCOVER TAPE
6,7,10125,MINI FUNKY DESIGN TAPES
7,8,10133,COLOURING PENCILS BROWN TUBE
8,9,10135,COLOURING PENCILS BROWN TUBE
9,10,11001,ASSTD DESIGN RACING CAR PEN



4. CONTRÔLE D'UNICITÉ
Nombre de lignes : 4026
Nombre de ProduitKey uniques : 4026
ProduitKey uniques : True
Couples StockCode / Description uniques : True
Clés nulles : 0

5. RECHERCHE DES CODES AMBIGUS
Nombre de codes associés à plusieurs descriptions : 212

Exemples de codes ambigus :


,ProduitKey,StockCode_norm,Description_norm
42,43,16156L,WRAP CAROUSEL
43,44,16156L,"WRAP, CAROUSEL"
101,102,17107D,FLOWER FAIRY 5 DRAWER LINERS
102,103,17107D,FLOWER FAIRY 5 SUMMER DRAW LINERS
103,104,17107D,"FLOWER FAIRY,5 SUMMER B'DRAW LINERS"
121,122,20622,VIP PASSPORT COVER
122,123,20622,VIPPASSPORT COVER
174,175,20725,LUNCH BAG RED RETROSPOT
175,176,20725,LUNCH BAG RED SPOTTY
270,271,20914,SET/5 RED RETROSPOT LID GLASS BOWLS



6. CONTRÔLE DE COUVERTURE
Couples distincts utilisés par les ventes : 4026
Couples rattachés à DimProduit : 4026
Couples non rattachés : 0

7. PREUVE DE L'ÉTAPE 06
Variable obtenue : dim_produit
Grain : un couple StockCode / Description normalisée
Clé primaire : ProduitKey
Clé de rapprochement : StockCode_norm + Description_norm

RÉSULTAT DES CONTRÔLES
ProduitKey unique : True
Couple unique : True
Couverture complète : True

DÉCISION
Aucune vente n'a été supprimée.
Aucune catégorie ou marque n'a été inventée.
La dimension est construite à partir des valeurs distinctes.


### Preuve de l’étape — à renseigner après exécution
- **Variables ou tables obtenues :**
  - dim_produit : dimension produit contenant *4 026 produits distincts*.
  - ProduitKey : clé primaire attribuée à chaque produit.
  - StockCode_norm : code produit normalisé.
  - Description_norm : description du produit normalisée.

- **Résultat du contrôle / observations :**
  - 522 364 lignes de ventes examinées.
  - 4 026 couples distincts StockCode_norm / Description_norm.
  - 4 026 clés ProduitKey uniques.
  - Aucune clé nulle.
  - 212 codes associés à plusieurs descriptions différentes.
  - 4 026 couples correctement rattachés à DimProduit.
  - 0 couple non rattaché.
- **Décision ou correction :**
Nous avons conservé le couple StockCode_norm et Description_norm comme clé de rapprochement afin d'éviter les ambiguïtés liées aux codes associés à plusieurs descriptions. Les descriptions absentes et les codes absents sont inexistants dans nos données. Aucune vente n'a été supprimée et aucune catégorie ou marque n'a été inventée. Les clés ont été attribuées selon un ordre déterministe.

# ÉTAPE 07 — Construire DimClient et gérer Client inconnu
**Durée indicative : 15 min · Objectif :** Conserver toute vente admissible même si CustomerID manque.



## Questions du TP
1. Comment gérerez-vous CustomerID manquant sans créer de clé orpheline ?
2. Créer une ligne technique « Client inconnu » dans DimClient puis prévoir le rattachement des ventes concernées.
3. Pourquoi supprimer ces lignes aurait-il un impact sur le CA et les commandes ?

## Question complémentaire — prendre du recul
ClientKey = 0 signifie-t-il que le véritable CustomerID vaut 0 ? Comment distinguer une absence initiale d’identifiant et un lookup échoué pour un client pourtant renseigné ?

### Réponses du binôme — à compléter

**Convention d’identifiant absent :** CustomerID est considéré comme absent lorsqu'il contient une valeur nulle ou une chaîne vide après suppression des espaces 
**Clé réservée au membre inconnu :** 0. 
**Statut et valeur de CustomerID de cette ligne :** Le statut est « Client inconnu » et CustomerID reste vide (valeur NaN).

**Provenances des ventes initialement sans client à préserver :** Les ventes sans identifiant client sont conservées grâce à leur SourceIndex, qui permet de retrouver leur provenance dans L2. 
**Contrôle prévu pour un client renseigné non retrouvé :** Vérifier les identifiants renseignés qui ne correspondent à aucune ligne de DimClient et les signaler comme des erreurs de rattachement.

**Notre justification :**  
Nous avons choisi de créer une clé technique réservée au client inconnu afin de préserver toutes les ventes admissibles. Cette méthode permet de conserver le chiffre d'affaires et les quantités vendues sans inventer un identifiant client.

**Point à vérifier / difficulté rencontrée :**  
Nous devons vérifier que les clients identifiés possèdent des clés uniques, que le membre inconnu existe une seule fois et que toutes les ventes sans CustomerID sont correctement conservées.

In [9]:
## CODE PYTHON — ÉTAPE 07

import pandas as pd
import numpy as np
from IPython.display import display

# 1. Vérifier les colonnes nécessaires

colonnes_requises = ["SourceIndex", "CustomerID_norm"]

for col in colonnes_requises:
    if col not in df.columns:
        raise ValueError(f"Colonne absente : {col}")

# 2. Normaliser les identifiants sans modifier df_prep

df["CustomerID_norm"] = (
    df["CustomerID_norm"]
    .astype("string")
    .str.strip()
)

# Transformer les chaînes vides en valeurs manquantes
df["CustomerID_norm"] = (
    df["CustomerID_norm"].replace("", pd.NA)
)

# 3. Identifier les ventes sans client AVANT le lookup

ventes_sans_client = df[
    df["CustomerID_norm"].isna()
].copy()

sourceindex_sans_client = ventes_sans_client["SourceIndex"].copy()

print("Nombre de ventes sans identifiant client :",
      len(ventes_sans_client))

# 4. Extraire les clients identifiés distincts

clients_identifies = (
    df.loc[
        df["CustomerID_norm"].notna(),
        ["CustomerID_norm"]
    ]
    .drop_duplicates()
    .sort_values("CustomerID_norm")
    .reset_index(drop=True)
)

# 5. Construire DimClient

dim_client = clients_identifies.copy()

# Attribution des clés à partir de 1
dim_client.insert(
    0,
    "ClientKey",
    range(1, len(dim_client) + 1)
)

# Ajouter le statut d'identification
dim_client["StatutIdentification"] = "Client identifié"

# 6. Ajouter exactement un membre Client inconnu

client_inconnu = pd.DataFrame({
    "ClientKey": [0],
    "CustomerID_norm": [pd.NA],
    "StatutIdentification": ["Client inconnu"]
})

dim_client = pd.concat(
    [client_inconnu, dim_client],
    ignore_index=True
)

# 7. Vérifier les informations de DimClient

print("\n===== DIMENSION CLIENT =====")

print("Nombre total de clients :", len(dim_client))

print("\nExtrait de DimClient :")
display(dim_client.head(10))

print("\nMembre Client inconnu :")
display(dim_client[dim_client["ClientKey"] == 0])

# 8. Contrôler l'unicité des clés

print("\n===== CONTRÔLE DES CLÉS =====")

print("Clés uniques :",
      dim_client["ClientKey"].is_unique)

print("Nombre de clés nulles :",
      dim_client["ClientKey"].isna().sum())

print("Nombre de membres inconnus :",
      (dim_client["ClientKey"] == 0).sum())

# 9. Vérifier les identifiants renseignés

clients_lookup = dim_client[
    dim_client["CustomerID_norm"].notna()
].set_index("CustomerID_norm")["ClientKey"]

df["ClientKey"] = (
    df["CustomerID_norm"].map(clients_lookup)
)

# Affecter 0 uniquement aux identifiants initialement absents
df.loc[
    df["CustomerID_norm"].isna(),
    "ClientKey"
] = 0

# 10. Détecter les lookups échoués

lookup_echoue = (
    df["CustomerID_norm"].notna()
    & df["ClientKey"].isna()
)

print("\n===== CONTRÔLE DES LOOKUPS =====")

print("Ventes sans client initialement :",
      len(ventes_sans_client))

print("Ventes avec lookup échoué :",
      lookup_echoue.sum())

print("Ventes rattachées à Client inconnu :",
      (df["ClientKey"] == 0).sum())

print("Ventes avec une clé client valide :",
      df["ClientKey"].notna().sum())

# 11. Vérifier la conservation des ventes sans client

controle_sourceindex = (
    ventes_sans_client["SourceIndex"].equals(
        df.loc[
            df["ClientKey"] == 0,
            "SourceIndex"
        ].reset_index(drop=True)
    )
)

# 12. Résumé des contrôles

print("\n===== RÉSULTAT FINAL =====")

print("Nombre de clients identifiés :",
      len(clients_identifies))

print("Nombre total de membres DimClient :",
      len(dim_client))

print("Clé 0 réservée :",
      (dim_client["ClientKey"] == 0).sum() == 1)

print("Conservation des ventes sans client :",
      controle_sourceindex)

print("Lookup échoué :", lookup_echoue.sum())

if lookup_echoue.sum() == 0 and controle_sourceindex:
    print("\nVALIDATION : DimClient est construite correctement.")
else:
    print("\nATTENTION : vérifier les anomalies de rattachement.")

print("\nÉTAPE 07 TERMINÉE.")

Nombre de ventes sans identifiant client : 131355

===== DIMENSION CLIENT =====
Nombre total de clients : 4335

Extrait de DimClient :


C:\Users\pc\AppData\Local\Temp\ipykernel_19232\3614183437.py:73: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  dim_client = pd.concat(


,ClientKey,CustomerID_norm,StatutIdentification
0,0,<NA>,Client inconnu
1,1,12346.0,Client identifié
2,2,12347.0,Client identifié
3,3,12348.0,Client identifié
4,4,12349.0,Client identifié
5,5,12350.0,Client identifié
6,6,12352.0,Client identifié
7,7,12353.0,Client identifié
8,8,12354.0,Client identifié
9,9,12355.0,Client identifié



Membre Client inconnu :


,ClientKey,CustomerID_norm,StatutIdentification
0,0,<NA>,Client inconnu



===== CONTRÔLE DES CLÉS =====
Clés uniques : True
Nombre de clés nulles : 0
Nombre de membres inconnus : 1

===== CONTRÔLE DES LOOKUPS =====
Ventes sans client initialement : 131355
Ventes avec lookup échoué : 0
Ventes rattachées à Client inconnu : 131355
Ventes avec une clé client valide : 522364

===== RÉSULTAT FINAL =====
Nombre de clients identifiés : 4334
Nombre total de membres DimClient : 4335
Clé 0 réservée : True
Conservation des ventes sans client : False
Lookup échoué : 0

ATTENTION : vérifier les anomalies de rattachement.

ÉTAPE 07 TERMINÉE.


### Preuve de l’étape — à renseigner après exécution
- **Variables ou tables obtenues :** Dimension DimClient contenant 4 335 membres, dont 4 334 clients identifiés et un membre spécial « Client inconnu » avec la clé ClientKey = 0.
- **Résultat du contrôle / observations :**
- 131 355 ventes ne possèdent pas d'identifiant client dans les données sources.
- Les clés primaires sont uniques et aucune clé n'est nulle.
- Aucun échec de lookup n'a été détecté.
- Les 131 355 ventes sans identifiant client sont rattachées au membre « Client inconnu ».
- Une anomalie est signalée : le contrôle de conservation des ventes sans client retourne False.
- **Décision ou correction :** Conserver le membre spécial « Client inconnu » avec la clé 0 pour représenter les clients non identifiés. Vérifier et corriger le contrôle de conservation afin de confirmer que les 131 355 ventes sont correctement rattachées à ce membre.

# ÉTAPE 08 — Construire DimPays et consolider les dimensions
**Durée indicative : 10 min · Objectif :** Préserver la géographie indépendamment de l’identification client.

**Repère :** CP4 — quatre dimensions cohérentes, clés uniques et schéma explicite.

## Questions du TP
1. Construire DimPays à partir de Country et générer une clé cohérente.
2. Pourquoi DimPays est-elle séparée de DimClient ?

## Question complémentaire — prendre du recul
Deux ventes sans CustomerID ont des pays différents. Peut-on attribuer un pays unique au membre Client inconnu sans perdre ou déformer l’information ? Justifiez votre modèle.

### Réponses du binôme — à compléter

**Grain de DimPays et règle de PaysKey :** …
Grain de DimPays et règle de PaysKey : Une ligne de DimPays représente un pays distinct. Chaque pays possède une clé primaire PaysKey unique, attribuée de manière déterministe à partir de 1.

| Dimension | Effectif calculé | Clés uniques ? | Valeurs de rapprochement uniques ? |
| --- | --- | --- | --- |
| DimDate | 305 | Oui | Oui |
| DimProduit | 4 026 | Oui | Oui |
| DimClient | 4 335 | Oui | Oui (pour les clients identifiés) |
| DimPays | À compléter après exécution | Oui (à vérifier) | Oui (à vérifier) |

**Notre justification :**  
Nous avons séparé DimPays de DimClient afin de préserver les informations géographiques indépendamment de l'identification des clients. Cette organisation permet de conserver toutes les ventes, y compris celles dont le CustomerID est absent. Les quatre dimensions permettent ainsi de construire un modèle en étoile cohérent

**Point à vérifier / difficulté rencontrée :**  
Nous devons vérifier l'unicité des PaysKey, l'absence de doublons dans Country_norm et la couverture de toutes les valeurs de pays utilisées dans les ventes. Nous devons également vérifier que les quatre dimensions possèdent des clés primaires uniques.

In [10]:
import pandas as pd
import numpy as np
from IPython.display import display

# ============================================================
# 1. EXAMEN DES DONNÉES PAYS
# ============================================================

print("===== 1. EXAMEN DES DONNÉES PAYS =====")

if "Country_norm" not in df.columns:
    raise ValueError("Country_norm est absente de df.")

print("Nombre de ventes :", len(df))
print("Nombre de pays manquants :",
      df["Country_norm"].isna().sum())

# Normalisation légère sans inventer de pays
df["Country_norm"] = (
    df["Country_norm"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
)

print("Nombre de pays distincts :",
      df["Country_norm"].nunique())

# ============================================================
# 2. CONSTRUCTION DE DIM_PAYS
# ============================================================

print("\n===== 2. CONSTRUCTION DE DIM_PAYS =====")

pays_distincts = (
    df.loc[df["Country_norm"].notna(), ["Country_norm"]]
    .drop_duplicates()
    .sort_values("Country_norm")
    .reset_index(drop=True)
)

dim_pays = pays_distincts.copy()

dim_pays.insert(
    0,
    "PaysKey",
    range(1, len(dim_pays) + 1)
)

print("Nombre de pays :", len(dim_pays))

print("\nExtrait de DimPays :")
display(dim_pays.head(10))

# ============================================================
# 3. CONTRÔLE DES CLÉS ET DES PAYS
# ============================================================

print("\n===== 3. CONTRÔLE DE DIM_PAYS =====")

print("PaysKey uniques :",
      dim_pays["PaysKey"].is_unique)

print("Country_norm unique :",
      dim_pays["Country_norm"].is_unique)

print("Clés nulles :",
      dim_pays["PaysKey"].isna().sum())

print("Pays manquants dans la dimension :",
      dim_pays["Country_norm"].isna().sum())

# ============================================================
# 4. RATTACHEMENT DES PAYS AUX VENTES
# ============================================================

print("\n===== 4. CONTRÔLE DE COUVERTURE =====")

lookup_pays = dim_pays.set_index(
    "Country_norm"
)["PaysKey"]

df["PaysKey"] = df["Country_norm"].map(lookup_pays)

pays_non_rattaches = (
    df["Country_norm"].notna()
    & df["PaysKey"].isna()
)

print("Ventes avec pays renseigné :",
      df["Country_norm"].notna().sum())

print("Ventes rattachées à DimPays :",
      df["PaysKey"].notna().sum())

print("Ventes avec pays non rattaché :",
      pays_non_rattaches.sum())

print("Ventes sans pays dans la source :",
      df["Country_norm"].isna().sum())

# ============================================================
# 5. CONSOLIDATION DES QUATRE DIMENSIONS
# ============================================================

print("\n===== 5. CONSOLIDATION DES DIMENSIONS =====")

# Contrôle DimDate
controle_date = {
    "Dimension": "DimDate",
    "Effectif": len(dim_date),
    "Clés uniques": dim_date["DateKey"].is_unique,
    "Valeurs de rapprochement uniques":
        dim_date["Date"].is_unique
}

# Contrôle DimProduit
controle_produit = {
    "Dimension": "DimProduit",
    "Effectif": len(dim_produit),
    "Clés uniques": dim_produit["ProduitKey"].is_unique,
    "Valeurs de rapprochement uniques":
        not dim_produit.duplicated(
            ["StockCode_norm", "Description_norm"]
        ).any()
}

# Contrôle DimClient
controle_client = {
    "Dimension": "DimClient",
    "Effectif": len(dim_client),
    "Clés uniques": dim_client["ClientKey"].is_unique,
    "Valeurs de rapprochement uniques":
        dim_client.loc[
            dim_client["CustomerID_norm"].notna(),
            "CustomerID_norm"
        ].is_unique
}

# Contrôle DimPays
controle_pays = {
    "Dimension": "DimPays",
    "Effectif": len(dim_pays),
    "Clés uniques": dim_pays["PaysKey"].is_unique,
    "Valeurs de rapprochement uniques":
        dim_pays["Country_norm"].is_unique
}

controles_dimensions = pd.DataFrame([
    controle_date,
    controle_produit,
    controle_client,
    controle_pays
])

display(controles_dimensions)

# ============================================================
# 6. CONTRÔLE DU MEMBRE CLIENT INCONNU
# ============================================================

print("\n===== 6. CONTRÔLE CLIENT INCONNU =====")

print("Nombre de membres inconnus :",
      (dim_client["ClientKey"] == 0).sum())

print("Ventes rattachées au client inconnu :",
      (df["ClientKey"] == 0).sum())

# ============================================================
# 7. BILAN FINAL
# ============================================================

print("\n===== 7. BILAN FINAL DE L'ÉTAPE 08 =====")

print("DimDate :", len(dim_date), "lignes")
print("DimProduit :", len(dim_produit), "lignes")
print("DimClient :", len(dim_client), "lignes")
print("DimPays :", len(dim_pays), "lignes")

toutes_cles_uniques = (
    controles_dimensions["Clés uniques"].all()
)

toutes_valeurs_uniques = (
    controles_dimensions[
        "Valeurs de rapprochement uniques"
    ].all()
)

print("\nToutes les clés sont uniques :",
      toutes_cles_uniques)

print("Toutes les valeurs de rapprochement sont uniques :",
      toutes_valeurs_uniques)

if (
    toutes_cles_uniques
    and toutes_valeurs_uniques
    and pays_non_rattaches.sum() == 0
):
    print("\nVALIDATION : les quatre dimensions sont consolidées.")
else:
    print("\nATTENTION : vérifier les anomalies signalées.")

print("\nÉTAPE 08 TERMINÉE.")

===== 1. EXAMEN DES DONNÉES PAYS =====
Nombre de ventes : 522364
Nombre de pays manquants : 0
Nombre de pays distincts : 38

===== 2. CONSTRUCTION DE DIM_PAYS =====
Nombre de pays : 38

Extrait de DimPays :


,PaysKey,Country_norm
0,1,AUSTRALIA
1,2,AUSTRIA
2,3,BAHRAIN
3,4,BELGIUM
4,5,BRAZIL
5,6,CANADA
6,7,CHANNEL ISLANDS
7,8,CYPRUS
8,9,CZECH REPUBLIC
9,10,DENMARK



===== 3. CONTRÔLE DE DIM_PAYS =====
PaysKey uniques : True
Country_norm unique : True
Clés nulles : 0
Pays manquants dans la dimension : 0

===== 4. CONTRÔLE DE COUVERTURE =====
Ventes avec pays renseigné : 522364
Ventes rattachées à DimPays : 522364
Ventes avec pays non rattaché : 0
Ventes sans pays dans la source : 0

===== 5. CONSOLIDATION DES DIMENSIONS =====


,Dimension,Effectif,Clés uniques,Valeurs de rapprochement uniques
0,DimDate,305,True,True
1,DimProduit,4026,True,True
2,DimClient,4335,True,True
3,DimPays,38,True,True



===== 6. CONTRÔLE CLIENT INCONNU =====
Nombre de membres inconnus : 1
Ventes rattachées au client inconnu : 131355

===== 7. BILAN FINAL DE L'ÉTAPE 08 =====
DimDate : 305 lignes
DimProduit : 4026 lignes
DimClient : 4335 lignes
DimPays : 38 lignes

Toutes les clés sont uniques : True
Toutes les valeurs de rapprochement sont uniques : True

VALIDATION : les quatre dimensions sont consolidées.

ÉTAPE 08 TERMINÉE.


### Preuve de l’étape — à renseigner après exécution
- **Variables ou tables obtenues :** Les quatre dimensions ont été construites et consolidées : dim_date (305 lignes), dim_produit (4 026 lignes), dim_client (4 335 lignes) et dim_pays (38 lignes).

- **Résultat du contrôle / observations :** Toutes les clés primaires sont uniques et aucune clé nulle n'est présente. Les valeurs de rapprochement sont également uniques. Les 522 364 ventes sont rattachées à une valeur de pays valide. Aucun pays n'est manquant ou non rattaché. Le membre « Client inconnu » (ClientKey = 0) est bien conservé pour les 131 355 ventes sans identifiant client.

- **Décision ou correction :** Les quatre dimensions sont validées. Aucune vente n'a été supprimée et aucun pays n'a été inventé. Le pays reste indépendant du client afin de conserver correctement les informations géographiques, même pour les ventes dont le client est inconnu. Les dimensions sont prêtes pour le rattachement à la table de faits FactVente.

# ÉTAPE 09 — Rechercher les clés — lookups contrôlés
**Durée indicative : 25 min · Objectif :** Rattacher chaque vente à une seule référence de chaque dimension.



## Questions du TP
1. Rattacher les clés des dimensions en respectant le grain.
2. Toutes les lignes de vente disposent-elles d’une clé client valide ?

## Question complémentaire — prendre du recul
Quelle différence entre une jointure gauche et une jointure interne pour une référence absente ? À quoi sert validate="many_to_one" ? Pourquoi garder le même effectif ne suffit-il pas à prouver que les bonnes ventes sont conservées ?

### Réponses du binôme — à compléter


| Lookup | Colonnes de rapprochement | Clé récupérée | Contrôle prévu |
| --- | --- | --- | --- |
| Date | Date au jour (Date) | DateKey | Vérifier que chaque vente possède une date correspondante dans DimDate. |
| Produit | StockCode_norm + Description_norm | ProduitKey | Vérifier l'unicité du couple et l'absence de produits non rattachés. |
| Client | CustomerID_norm | ClientKey | Vérifier les clients retrouvés et affecter la clé 0 uniquement aux clients initialement inconnus. |
| Pays | Country_norm | PaysKey | Vérifier que chaque pays est présent dans DimPays et qu'aucune clé ne manque. |

**Exemple suivi : valeur source → membre retrouvé → clé copiée :** 
| Dimension | Valeur source | Membre retrouvé | Clé copiée |
| --- | --- | --- | --- |
| Date | 2010-12-01 | 1 décembre 2010 | 20101201 |
| Produit | 85123A | Produit correspondant | ProduitKey correspondant |
| Client | 12347 | Client identifié 12347 | ClientKey correspondant |
| Pays | AUSTRALIA | AUSTRALIA | PaysKey correspondant |
| Client inconnu | Valeur absente | Client inconnu | 0 |

**Notre justification :**  
Nous utilisons des jointures gauches (left) pour conserver toutes les ventes, même lorsqu'une correspondance est absente. Le paramètre validate="many_to_one" permet de vérifier qu'une vente correspond à un seul membre de chaque dimension. Nous contrôlons également les clés récupérées, le nombre de lignes, les provenances et le montant total des ventes.

**Point à vérifier / difficulté rencontrée :**  
Vérifier que toutes les ventes sont correctement rattachées aux quatre dimensions, que les clés étrangères ne contiennent aucune valeur manquante et que les clients initialement sans identifiant sont associés à ClientKey = 0. Les clients renseignés mais non retrouvés doivent être signalés et corrigés.

In [12]:
import pandas as pd

# ============================================================
# ÉTAPE 09 — RECHERCHER LES CLÉS : LOOKUPS CONTRÔLÉS
# ============================================================

# ------------------------------------------------------------
# 1. RÉINITIALISATION
# ------------------------------------------------------------

# Repartir des données préparées pour éviter les anciennes clés
df_rattache = df.copy()

# Vérifier les colonnes nécessaires
colonnes_requises = [
    "SourceIndex",
    "InvoiceDate_horodatage",
    "StockCode_norm",
    "Description_norm",
    "CustomerID_norm",
    "Country_norm",
    "MontantLigne"
]

for col in colonnes_requises:
    if col not in df_rattache.columns:
        raise KeyError(f"Colonne absente dans df : {col}")

effectif_initial = len(df_rattache)
montant_initial = df_rattache["MontantLigne"].sum()
sources_initiales = set(df_rattache["SourceIndex"])

journal = []

print("Nombre de ventes initiales :", effectif_initial)
print("Montant total initial :", montant_initial)


# ------------------------------------------------------------
# 2. FONCTION DE CONTRÔLE
# ------------------------------------------------------------

def controler(etape, df_avant, colonne_cle):

    effectif = len(df_rattache)
    montant = df_rattache["MontantLigne"].sum()

    cles_manquantes = df_rattache[colonne_cle].isna().sum()

    journal.append({
        "Lookup": etape,
        "Lignes avant": len(df_avant),
        "Lignes après": effectif,
        "Clés manquantes": int(cles_manquantes),
        "Montant total": montant,
        "Effectif conservé": effectif == effectif_initial,
        "Montant conservé": abs(
            montant - montant_initial
        ) < 0.01
    })

    print(f"\n===== CONTRÔLE {etape} =====")
    print("Lignes avant :", len(df_avant))
    print("Lignes après :", effectif)
    print("Clés manquantes :", cles_manquantes)
    print("Montant total :", montant)


# ------------------------------------------------------------
# 3. LOOKUP DATE
# ------------------------------------------------------------

print("\n===== 1. LOOKUP DATE =====")

# Nettoyer les anciennes colonnes éventuelles
for col in ["Date", "DateKey"]:
    if col in df_rattache.columns:
        df_rattache.drop(columns=[col], inplace=True)

avant = df_rattache.copy()

# Conserver l'horodatage complet et extraire le jour
df_rattache["Date"] = pd.to_datetime(
    df_rattache["InvoiceDate_horodatage"],
    errors="coerce"
).dt.normalize()

date_lookup = dim_date[["Date", "DateKey"]].copy()

date_lookup["Date"] = pd.to_datetime(
    date_lookup["Date"],
    errors="coerce"
).dt.normalize()

# Contrôle de la dimension
assert date_lookup["Date"].is_unique
assert date_lookup["DateKey"].is_unique

# Jointure gauche
df_rattache = df_rattache.merge(
    date_lookup,
    on="Date",
    how="left",
    validate="many_to_one"
)

controler("Date", avant, "DateKey")


# ------------------------------------------------------------
# 4. LOOKUP PRODUIT
# ------------------------------------------------------------

print("\n===== 2. LOOKUP PRODUIT =====")

if "ProduitKey" in df_rattache.columns:
    df_rattache.drop(
        columns=["ProduitKey"],
        inplace=True
    )

avant = df_rattache.copy()

produit_lookup = dim_produit[
    ["StockCode_norm", "Description_norm", "ProduitKey"]
].copy()

assert not produit_lookup.duplicated(
    ["StockCode_norm", "Description_norm"]
).any()

df_rattache = df_rattache.merge(
    produit_lookup,
    on=["StockCode_norm", "Description_norm"],
    how="left",
    validate="many_to_one"
)

controler("Produit", avant, "ProduitKey")


# ------------------------------------------------------------
# 5. LOOKUP CLIENT
# ------------------------------------------------------------

print("\n===== 3. LOOKUP CLIENT =====")

if "ClientKey" in df_rattache.columns:
    df_rattache.drop(
        columns=["ClientKey"],
        inplace=True
    )

avant = df_rattache.copy()

# Normalisation des identifiants clients
client_source = (
    df_rattache["CustomerID_norm"]
    .astype("string")
    .str.strip()
)

# Identifier les clients initialement absents
client_absent = (
    client_source.isna() |
    client_source.eq("").fillna(False)
)

# Créer la colonne de rapprochement
df_rattache["_ClientLookup"] = client_source.mask(
    client_absent
)

client_lookup = dim_client[
    ["CustomerID_norm", "ClientKey"]
].copy()

client_lookup["_ClientLookup"] = (
    client_lookup["CustomerID_norm"]
    .astype("string")
    .str.strip()
)

# Exclure le membre inconnu du lookup classique
client_lookup = client_lookup[
    client_lookup["_ClientLookup"].notna()
    & client_lookup["_ClientLookup"].ne("")
].copy()

assert client_lookup["_ClientLookup"].is_unique

# Jointure gauche
df_rattache = df_rattache.merge(
    client_lookup[
        ["_ClientLookup", "ClientKey"]
    ],
    on="_ClientLookup",
    how="left",
    validate="many_to_one"
)

# Affecter la clé 0 uniquement aux identifiants absents
df_rattache.loc[
    client_absent,
    "ClientKey"
] = 0

# Identifier les clients renseignés mais non retrouvés
clients_non_retrouves = df_rattache[
    (~client_absent)
    & df_rattache["ClientKey"].isna()
].copy()

controler("Client", avant, "ClientKey")

print(
    "\nVentes initialement sans client :",
    client_absent.sum()
)

print(
    "Clients renseignés non retrouvés :",
    len(clients_non_retrouves)
)

if len(clients_non_retrouves) > 0:
    print("Exemples de clients non retrouvés :")
    display(
        clients_non_retrouves[
            ["SourceIndex", "CustomerID_norm"]
        ].head(20)
    )


# ------------------------------------------------------------
# 6. LOOKUP PAYS
# ------------------------------------------------------------

print("\n===== 4. LOOKUP PAYS =====")

if "PaysKey" in df_rattache.columns:
    df_rattache.drop(
        columns=["PaysKey"],
        inplace=True
    )

avant = df_rattache.copy()

pays_lookup = dim_pays[
    ["Country_norm", "PaysKey"]
].copy()

assert pays_lookup["Country_norm"].is_unique

df_rattache = df_rattache.merge(
    pays_lookup,
    on="Country_norm",
    how="left",
    validate="many_to_one"
)

controler("Pays", avant, "PaysKey")


# ------------------------------------------------------------
# 7. CONTRÔLE DES PROVENANCES
# ------------------------------------------------------------

print("\n===== 5. CONTRÔLE DES PROVENANCES =====")

sources_finales = set(df_rattache["SourceIndex"])

print("SourceIndex initiaux :", len(sources_initiales))
print("SourceIndex finaux :", len(sources_finales))

print(
    "Sources perdues :",
    len(sources_initiales - sources_finales)
)

print(
    "Sources supplémentaires :",
    len(sources_finales - sources_initiales)
)


# ------------------------------------------------------------
# 8. CONTRÔLE DES QUATRE CLÉS
# ------------------------------------------------------------

print("\n===== 6. CONTRÔLE DES CLÉS ÉTRANGÈRES =====")

colonnes_cles = [
    "DateKey",
    "ProduitKey",
    "ClientKey",
    "PaysKey"
]

for col in colonnes_cles:
    print(
        col,
        ":",
        df_rattache[col].isna().sum(),
        "clé(s) manquante(s)"
    )

ventes_non_rattachees = df_rattache[
    df_rattache[colonnes_cles].isna().any(axis=1)
]

print(
    "\nVentes non entièrement rattachées :",
    len(ventes_non_rattachees)
)

if len(ventes_non_rattachees) > 0:
    display(
        ventes_non_rattachees[
            ["SourceIndex"] + colonnes_cles
        ].head(20)
    )


# ------------------------------------------------------------
# 9. EXEMPLE D'UNE VENTE SUIVIE
# ------------------------------------------------------------

print("\n===== 7. EXEMPLE D'UNE VENTE =====")

colonnes_affichage = [
    "SourceIndex",
    "InvoiceNo",
    "InvoiceDate_horodatage",
    "StockCode_norm",
    "Description_norm",
    "CustomerID_norm",
    "Country_norm",
    "DateKey",
    "ProduitKey",
    "ClientKey",
    "PaysKey"
]

display(
    df_rattache[colonnes_affichage].head(1)
)


# ------------------------------------------------------------
# 10. JOURNAL DES RATTACHEMENTS
# ------------------------------------------------------------

print("\n===== 8. JOURNAL DES LOOKUPS =====")

df_journal = pd.DataFrame(journal)

display(df_journal)


# ------------------------------------------------------------
# 11. BILAN FINAL
# ------------------------------------------------------------

print("\n===== 9. BILAN FINAL DE L'ÉTAPE 09 =====")

print("Nombre de ventes initiales :", effectif_initial)
print("Nombre de ventes finales :", len(df_rattache))

print("Montant initial :", montant_initial)
print(
    "Montant final :",
    df_rattache["MontantLigne"].sum()
)

print(
    "Toutes les provenances conservées :",
    sources_initiales == sources_finales
)

print(
    "Toutes les ventes rattachées :",
    df_rattache[colonnes_cles].notna().all().all()
)

print(
    "Ventes avec Client inconnu :",
    (df_rattache["ClientKey"] == 0).sum()
)

# Validation finale
validation = (
    len(df_rattache) == effectif_initial
    and sources_initiales == sources_finales
    and df_rattache[colonnes_cles].notna().all().all()
    and abs(
        df_rattache["MontantLigne"].sum()
        - montant_initial
    ) < 0.01
)

print("\n===================================")
print("VALIDATION DE L'ÉTAPE 09 :", validation)
print("===================================")

if validation:
    print("SUCCÈS : toutes les ventes sont conservées.")
    print("Les quatre clés étrangères sont renseignées.")
else:
    print("ATTENTION : des anomalies restent à corriger.")
    print("Consulter les contrôles et les ventes non rattachées.")

# df_rattache contient les ventes avec leurs quatre clés.

Nombre de ventes initiales : 522364
Montant total initial : 10244661.18

===== 1. LOOKUP DATE =====

===== CONTRÔLE Date =====
Lignes avant : 522364
Lignes après : 522364
Clés manquantes : 0
Montant total : 10244661.18

===== 2. LOOKUP PRODUIT =====

===== CONTRÔLE Produit =====
Lignes avant : 522364
Lignes après : 522364
Clés manquantes : 0
Montant total : 10244661.18

===== 3. LOOKUP CLIENT =====

===== CONTRÔLE Client =====
Lignes avant : 522364
Lignes après : 522364
Clés manquantes : 0
Montant total : 10244661.18

Ventes initialement sans client : 131355
Clients renseignés non retrouvés : 0

===== 4. LOOKUP PAYS =====

===== CONTRÔLE Pays =====
Lignes avant : 522364
Lignes après : 522364
Clés manquantes : 0
Montant total : 10244661.18

===== 5. CONTRÔLE DES PROVENANCES =====
SourceIndex initiaux : 522364
SourceIndex finaux : 522364
Sources perdues : 0
Sources supplémentaires : 0

===== 6. CONTRÔLE DES CLÉS ÉTRANGÈRES =====
DateKey : 0 clé(s) manquante(s)
ProduitKey : 0 clé(s) manqu

,SourceIndex,InvoiceNo,InvoiceDate_horodatage,StockCode_norm,Description_norm,CustomerID_norm,Country_norm,DateKey,ProduitKey,ClientKey,PaysKey
0,0,536365,2010-12-01 08:26:00,85123A,WHITE HANGING HEART T-LIGHT HOLDER,17850.0,UNITED KINGDOM,20101201,3533,4013.0,36



===== 8. JOURNAL DES LOOKUPS =====


,Lookup,Lignes avant,Lignes après,Clés manquantes,Montant total,Effectif conservé,Montant conservé
0,Date,522364,522364,0,10244661.18,True,True
1,Produit,522364,522364,0,10244661.18,True,True
2,Client,522364,522364,0,10244661.18,True,True
3,Pays,522364,522364,0,10244661.18,True,True



===== 9. BILAN FINAL DE L'ÉTAPE 09 =====
Nombre de ventes initiales : 522364
Nombre de ventes finales : 522364
Montant initial : 10244661.18
Montant final : 10244661.18
Toutes les provenances conservées : True
Toutes les ventes rattachées : True
Ventes avec Client inconnu : 131355

VALIDATION DE L'ÉTAPE 09 : True
SUCCÈS : toutes les ventes sont conservées.
Les quatre clés étrangères sont renseignées.


### Preuve de l’étape — à renseigner après exécution
- **Variables ou tables obtenues :** La variable df_rattache contenant les 522 364 ventes avec leurs quatre clés étrangères (DateKey, ProduitKey, ClientKey, PaysKey). Le journal des contrôles journal a également été obtenu.
- **Résultat du contrôle / observations :** Les 522 364 ventes sont conservées après les quatre opérations de lookup. Le montant total reste inchangé à 10 244 661,18. Aucune clé étrangère n'est manquante et aucune provenance (SourceIndex) n'a été perdue. Les 131 355 ventes initialement sans identifiant client sont correctement rattachées au membre « Client inconnu ». Aucun client renseigné n'est resté sans correspondance.
- **Décision ou correction :** Les quatre lookups sont validés. Les jointures gauches (how="left") avec validate="many_to_one" ont permis de conserver toutes les ventes et de contrôler l'unicité des correspondances. Aucune vente n'a été supprimée. La table df_rattache est prête pour la construction de la table de faits FactVente.

# ÉTAPE 10 — Construire FactVente au grain validé
**Durée indicative : 20 min · Objectif :** Organiser les observations, mesures et références dans le fait.



## Questions du TP
1. Construire FactVente avec les clés des dimensions et les mesures QuantitéVendue, PrixUnitaire et MontantLigne.
2. Pourquoi ne crée-t-on pas une DimCommande dans le MVP ?
3. Comment contrôler le nombre de lignes, les clés orphelines et les mesures ?

## Question complémentaire — prendre du recul
Une clé technique de fait suffit-elle à corriger un mélange de grains ? Deux ventes peuvent-elles partager les quatre clés dimensionnelles sans être des doublons ? Donnez une justification.

### Réponses du binôme — à compléter

**Colonnes retenues dans FactVente :** FactVenteKey, DateKey, ProduitKey, ClientKey, PaysKey, InvoiceNo, QuantiteVendue, PrixUnitaire, MontantLigne, SourceIndex

**Règle d’identification et provenance d’une ligne :** FactVenteKey est une clé technique unique. SourceIndex identifie la ligne d'origine.

| Nom source | Nom dans le fait | Rôle |
|---|---|---|
| Quantity / colonne dérivée validée | QuantiteVendue | Mesure : quantité vendue |
| UnitPrice / colonne dérivée validée | PrixUnitaire | Mesure : prix unitaire |
| MontantLigne | MontantLigne | Mesure : montant de la ligne |
| InvoiceNo | InvoiceNo | Dimension dégénérée : numéro de facture |

**Lieu de conservation de l’horodatage :** Dans df_rattache et dans FactVente si la colonne est disponible.

**Notre justification :**  
FactVente est construite au grain d'une ligne de vente d'un produit sur une facture. Elle contient les quatre clés étrangères permettant de relier les ventes aux dimensions, ainsi que les mesures nécessaires à l'analyse commerciale. InvoiceNo reste directement dans la table de faits car il s'agit d'une dimension dégénérée. Aucune agrégation ni suppression de vente n'est effectuée.

**Point à vérifier / difficulté rencontrée :**  
Vérifier l'unicité de FactVenteKey, l'absence de clés étrangères orphelines, la conservation des 522 364 lignes de vente et la stabilité du montant total de 10 244 661,18.

In [14]:
# ============================================================
# ÉTAPE 10 — CONSTRUIRE FACTVENTE AU GRAIN VALIDÉ
# ============================================================
# 1.
import pandas as pd

# ------------------------------------------------------------
# 2. VÉRIFICATION DES COLONNES NÉCESSAIRES
# ------------------------------------------------------------

colonnes_requises = [
    "DateKey",
    "ProduitKey",
    "ClientKey",
    "PaysKey",
    "InvoiceNo",
    "Quantity",
    "UnitPrice",
    "MontantLigne",
    "SourceIndex"
]

colonnes_absentes = [
    col for col in colonnes_requises
    if col not in df_rattache.columns
]

if colonnes_absentes:
    raise ValueError(
        f"Colonnes manquantes dans df_rattache : {colonnes_absentes}"
    )

print("\n1. CONTRÔLE DES DONNÉES SOURCE")
print("Nombre de ventes dans df_rattache :", len(df_rattache))
print("Colonnes nécessaires disponibles : True")

# ------------------------------------------------------------
# 3. CONSTRUCTION DE FACTVENTE
# ------------------------------------------------------------

# Copie des données : aucune modification de df_rattache
fact_vente = df_rattache.copy()

# Renommer les mesures selon le modèle décisionnel
fact_vente = fact_vente.rename(columns={
    "Quantity": "QuantiteVendue",
    "UnitPrice": "PrixUnitaire"
})

# Colonnes à conserver dans la table de faits
colonnes_fact = [
    "DateKey",
    "ProduitKey",
    "ClientKey",
    "PaysKey",
    "InvoiceNo",
    "QuantiteVendue",
    "PrixUnitaire",
    "MontantLigne",
    "SourceIndex"
]

# Conserver également les colonnes de provenance si disponibles
for col in ["SourceFile", "SourceRow"]:
    if col in fact_vente.columns:
        colonnes_fact.append(col)

# Conserver l'horodatage complet s'il existe
if "InvoiceDate_horodatage" in fact_vente.columns:
    colonnes_fact.append("InvoiceDate_horodatage")

# Construire la table de faits sans agrégation
fact_vente = fact_vente[colonnes_fact].copy()

# Ajouter une clé technique unique pour chaque observation
fact_vente.insert(
    0,
    "FactVenteKey",
    range(1, len(fact_vente) + 1)
)

print("\n2. CONSTRUCTION DE FACTVENTE")
print("Nombre de lignes :", len(fact_vente))
print("Nombre de colonnes :", len(fact_vente.columns))

# ------------------------------------------------------------
# 4. CONTRÔLE DES CLÉS PRIMAIRES
# ------------------------------------------------------------

print("\n3. CONTRÔLE DE LA CLÉ PRIMAIRE")

print(
    "FactVenteKey uniques :",
    fact_vente["FactVenteKey"].is_unique
)

print(
    "FactVenteKey nulles :",
    fact_vente["FactVenteKey"].isna().sum()
)

# Contrôle de la provenance
print(
    "SourceIndex uniques :",
    fact_vente["SourceIndex"].is_unique
)

# ------------------------------------------------------------
# 5. CONTRÔLE DES CLÉS ÉTRANGÈRES
# ------------------------------------------------------------

cles_etrangeres = [
    "DateKey",
    "ProduitKey",
    "ClientKey",
    "PaysKey"
]

print("\n4. CONTRÔLE DES CLÉS ÉTRANGÈRES")

dimensions = {
    "DateKey": dim_date,
    "ProduitKey": dim_produit,
    "ClientKey": dim_client,
    "PaysKey": dim_pays
}

for cle in cles_etrangeres:

    nb_nulles = fact_vente[cle].isna().sum()

    valeurs_valides = set(
        dimensions[cle][cle].dropna()
    )

    nb_orphelines = (
        ~fact_vente[cle].isin(valeurs_valides)
    ).sum()

    print(f"\n{cle} :")
    print("Clés nulles :", nb_nulles)
    print("Clés orphelines :", nb_orphelines)

# ------------------------------------------------------------
# 6. CONTRÔLE DES MESURES
# ------------------------------------------------------------

print("\n5. CONTRÔLE DES MESURES")

print(
    "Total QuantiteVendue :",
    fact_vente["QuantiteVendue"].sum()
)

print(
    "Montant total des ventes :",
    round(fact_vente["MontantLigne"].sum(), 2)
)

print(
    "PrixUnitaire manquants :",
    fact_vente["PrixUnitaire"].isna().sum()
)

print(
    "MontantLigne manquants :",
    fact_vente["MontantLigne"].isna().sum()
)

# Comparaison avec les données initiales
montant_initial = df_rattache["MontantLigne"].sum()
montant_final = fact_vente["MontantLigne"].sum()

print(
    "Montant initial :",
    round(montant_initial, 2)
)

print(
    "Montant final :",
    round(montant_final, 2)
)

print(
    "Montant conservé :",
    abs(montant_initial - montant_final) < 0.01
)

# ------------------------------------------------------------
# 7. CONTRÔLE DU GRAIN ET DE LA PROVENANCE
# ------------------------------------------------------------

print("\n6. CONTRÔLE DU GRAIN")

print("Grain : une ligne de vente d'un produit sur une facture.")

print(
    "Nombre de ventes initiales :",
    len(df_rattache)
)

print(
    "Nombre de lignes dans FactVente :",
    len(fact_vente)
)

print(
    "SourceIndex conservés :",
    fact_vente["SourceIndex"].nunique()
)

print(
    "Toutes les provenances conservées :",
    set(df_rattache["SourceIndex"])
    == set(fact_vente["SourceIndex"])
)

# ------------------------------------------------------------
# 8. AFFICHAGE D'UN EXTRAIT DE FACTVENTE
# ------------------------------------------------------------

print("\n7. EXTRAIT DE FACTVENTE")

display(fact_vente.head(10))

# ------------------------------------------------------------
# 9. BILAN FINAL
# ------------------------------------------------------------

cles_valides = all(
    fact_vente[cle].notna().all()
    and fact_vente[cle].isin(
        dimensions[cle][cle]
    ).all()
    for cle in cles_etrangeres
)

validation = (
    fact_vente["FactVenteKey"].is_unique
    and fact_vente["SourceIndex"].is_unique
    and len(fact_vente) == len(df_rattache)
    and cles_valides
    and abs(montant_initial - montant_final) < 0.01
    and set(df_rattache["SourceIndex"])
        == set(fact_vente["SourceIndex"])
)

print("\n" + "=" * 65)
print("8. BILAN FINAL DE L'ÉTAPE 10")
print("=" * 65)

print("Nombre de lignes :", len(fact_vente))
print("Nombre de clés primaires :", fact_vente["FactVenteKey"].nunique())
print("Toutes les clés étrangères valides :", cles_valides)
print("Toutes les provenances conservées :",
      set(df_rattache["SourceIndex"]) == set(fact_vente["SourceIndex"]))
print("Montant total conservé :", abs(montant_initial - montant_final) < 0.01)

print("\nVALIDATION DE L'ÉTAPE 10 :", validation)

if validation:
    print("SUCCÈS : FactVente est construite et validée.")
else:
    print("ATTENTION : vérifier les contrôles avant de valider.")


1. CONTRÔLE DES DONNÉES SOURCE
Nombre de ventes dans df_rattache : 522364
Colonnes nécessaires disponibles : True

2. CONSTRUCTION DE FACTVENTE
Nombre de lignes : 522364
Nombre de colonnes : 13

3. CONTRÔLE DE LA CLÉ PRIMAIRE
FactVenteKey uniques : True
FactVenteKey nulles : 0
SourceIndex uniques : True

4. CONTRÔLE DES CLÉS ÉTRANGÈRES

DateKey :
Clés nulles : 0
Clés orphelines : 0

ProduitKey :
Clés nulles : 0
Clés orphelines : 0

ClientKey :
Clés nulles : 0
Clés orphelines : 0

PaysKey :
Clés nulles : 0
Clés orphelines : 0

5. CONTRÔLE DES MESURES
Total QuantiteVendue : 5561117
Montant total des ventes : 10244661.18
PrixUnitaire manquants : 0
MontantLigne manquants : 0
Montant initial : 10244661.18
Montant final : 10244661.18
Montant conservé : True

6. CONTRÔLE DU GRAIN
Grain : une ligne de vente d'un produit sur une facture.
Nombre de ventes initiales : 522364
Nombre de lignes dans FactVente : 522364
SourceIndex conservés : 522364
Toutes les provenances conservées : True

7. EXTRA

,FactVenteKey,DateKey,ProduitKey,ClientKey,PaysKey,InvoiceNo,QuantiteVendue,PrixUnitaire,MontantLigne,SourceIndex,SourceFile,SourceRow,InvoiceDate_horodatage
0,1,20101201,3533,4013.0,36,536365,6,2.55,15.30,0,Online_Retail.csv,2,2010-12-01 08:26:00
1,2,20101201,2912,4013.0,36,536365,6,3.39,20.34,1,Online_Retail.csv,3,2010-12-01 08:26:00
2,3,20101201,3131,4013.0,36,536365,8,2.75,22.00,2,Online_Retail.csv,4,2010-12-01 08:26:00
3,4,20101201,3076,4013.0,36,536365,6,3.39,20.34,3,Online_Retail.csv,5,2010-12-01 08:26:00
4,5,20101201,3075,4013.0,36,536365,6,3.39,20.34,4,Online_Retail.csv,6,2010-12-01 08:26:00
5,6,20101201,1657,4013.0,36,536365,2,7.65,15.30,5,Online_Retail.csv,7,2010-12-01 08:26:00
6,7,20101201,779,4013.0,36,536365,6,4.25,25.50,6,Online_Retail.csv,8,2010-12-01 08:26:00
7,8,20101201,1542,4013.0,36,536366,6,1.85,11.10,7,Online_Retail.csv,9,2010-12-01 08:28:00
8,9,20101201,1540,4013.0,36,536366,6,1.85,11.10,8,Online_Retail.csv,10,2010-12-01 08:28:00
9,10,20101201,3347,535.0,36,536367,32,1.69,54.08,9,Online_Retail.csv,11,2010-12-01 08:34:00



8. BILAN FINAL DE L'ÉTAPE 10
Nombre de lignes : 522364
Nombre de clés primaires : 522364
Toutes les clés étrangères valides : True
Toutes les provenances conservées : True
Montant total conservé : True

VALIDATION DE L'ÉTAPE 10 : True
SUCCÈS : FactVente est construite et validée.


### Preuve de l’étape — à renseigner après exécution
- **Variables ou tables obtenues :** La table fact_vente a été construite avec 522 364 lignes et 13 colonnes. Elle contient les clés primaires et étrangères (FactVenteKey, DateKey, ProduitKey, ClientKey, PaysKey), le numéro de facture (InvoiceNo), les mesures (QuantiteVendue, PrixUnitaire, MontantLigne) ainsi que les informations de provenance (SourceIndex, SourceFile, SourceRow) et l'horodatage complet (InvoiceDate_horodatage)

- **Résultat du contrôle / observations :** La clé primaire FactVenteKey est unique et ne contient aucune valeur nulle. Les quatre clés étrangères sont valides, sans clé nulle ni orpheline. Les 522 364 ventes initiales sont conservées. La quantité totale vendue est de 5 561 117 unités et le montant total des ventes est de 10 244 661,18. Aucune mesure n'est manquante. Toutes les provenances sont conservées et le montant initial est égal au montant final.

- **Décision ou correction :** La table FactVente est validée. Aucune vente n'a été supprimée ni agrégée. Le grain d'une ligne de vente d'un produit sur une facture est respecté. Les clés étrangères permettent de relier correctement les quatre dimensions. La table est prête pour les prochaines étapes du Data Warehouse.

# ÉTAPE 11 — Contrôler et réconcilier L3
**Durée indicative : 30 min · Objectif :** Prouver la conservation des ventes et la validité des références.

**Repère :** CP5 — preuves de conservation et d’intégrité, pas seulement absence d’erreur Python.

## Questions du TP
1. Quels tests exécuterez-vous avant d’accepter L3 ?
2. Quels contrôles prouveront que le fait respecte son grain ?
3. Vérifier les clés uniques, les clés orphelines, le périmètre et la réconciliation de MontantLigne.

## Question complémentaire — prendre du recul
Deux erreurs peuvent-elles se compenser et donner le bon total ? Comment vérifier autre chose que le seul effectif et la seule somme ? Quel contrôle prouve spécifiquement la conservation des ventes au client inconnu ?

### Réponses du binôme — à compléter

| Contrôle | Résultat calculé | Attendu / critère | Conclusion / correction |
| --- | --- | --- | --- |
| Grain du fait | 522 364 lignes, une ligne de vente d'un produit sur une facture | Explicite et cohérent avec les observations | Grain respecté, aucune agrégation effectuée. |
| Clés DimDate | 305 clés uniques, aucune clé nulle | Uniques | Conforme, chaque clé représente un jour. |
| Clés DimProduit | 4 026 clés uniques, aucune clé nulle | Uniques | Conforme, chaque produit est identifié par son code et sa description normalisée. |
| Clés DimClient | Clés uniques, avec une clé réservée 0 pour Client inconnu | Uniques | Conforme, le client inconnu est représenté par une clé technique. |
| Clés DimPays | Clés uniques, attribuées aux pays distincts | Uniques | Conforme si le contrôle d'unicité et d'absence de valeurs nulles est validé. |
| Clés orphelines par dimension | DateKey : 0 ; ProduitKey : 0 ; ClientKey : 0 ; PaysKey : 0 | Aucune | Conforme, toutes les clés étrangères correspondent aux dimensions. |
| Lignes hors périmètre | 0 ligne supprimée pendant la construction de FactVente | Aucune | Conforme, les 522 364 ventes sont conservées. |
| Provenances L2 / L3 | 522 364 SourceIndex uniques et conservés | Mêmes observations, une fois chacune | Conforme, toutes les provenances L2 sont retrouvées dans L3. |
| Ventes au client inconnu | À confirmer par comparaison des SourceIndex des ventes sans CustomerID dans L2 et L3 | Mêmes provenances admissibles | Les ventes sans identifiant client doivent être conservées et rattachées à ClientKey = 0. |
| Quantité totale | 5 561 117 unités | L2 et L3 réconciliés | Conforme, la quantité totale est conservée. |
| Montants par ligne et en total | 522 364 montants cohérents ; total : 10 244 661,18 | L2 et L3 réconciliés | Conforme, les montants par ligne et le montant total sont conservés. |

### Conclusion générale

La table FactVente est correctement construite et respecte le grain défini. Les quatre clés étrangères sont valides et aucune clé orpheline n'a été détectée. Les 522 364 lignes de ventes sont conservées, avec une quantité totale de 5 561 117 unités et un montant total de 10 244 661,18.

Les contrôles confirment la cohérence entre L2 et L3. Le modèle décisionnel est donc prêt pour les contrôles complémentaires et l'exportation.

Petite remarque : le nombre exact de clients inconnus et les contrôles détaillés de DimPays ne figurent pas dans les résultats communiqués. Il faudra les vérifier dans le notebook avant de considérer ces deux contrôles comme définitivement validés.


**Notre justification :**  
Nous avons choisi d'effectuer plusieurs contrôles complémentaires afin de garantir la fiabilité du modèle décisionnel L3. La comparaison des provenances permet de vérifier que toutes les ventes de L2 sont conservées une seule fois dans FactVente. Les contrôles des clés étrangères garantissent la cohérence des relations entre les faits et les dimensions. Enfin, la comparaison des quantités et des montants permet de vérifier que le chargement n'a pas modifié les données commerciales.

**Point à vérifier / difficulté rencontrée :**  
Nous devons vérifier particulièrement les ventes sans CustomerID, les éventuelles clés orphelines et les écarts entre les montants de L2 et L3. Il faut également vérifier les provenances individuellement, car un total identique ne garantit pas à lui seul que toutes les ventes ont été correctement conservées.

In [15]:
# ============================================================
# ÉTAPE 11 — CONTRÔLER ET RÉCONCILIER L3
# TP3 - Validation du modèle décisionnel
# ============================================================

import pandas as pd
import numpy as np
from IPython.display import display

# ------------------------------------------------------------
# 1. Vérifier la présence des tables
# ------------------------------------------------------------

tables_requises = [
    "df_prep",
    "dim_date",
    "dim_produit",
    "dim_client",
    "dim_pays",
    "fact_vente"
]

for nom in tables_requises:
    if nom not in globals():
        raise NameError(f"Table absente : {nom}")

# Vérifier les colonnes obligatoires du fait

colonnes_fait = [
    "SourceIndex",
    "DateKey",
    "ProduitKey",
    "ClientKey",
    "PaysKey",
    "InvoiceNo",
    "QuantiteVendue",
    "PrixUnitaire",
    "MontantLigne"
]

for col in colonnes_fait:
    if col not in fact_vente.columns:
        raise ValueError(f"Colonne absente de fact_vente : {col}")

# ------------------------------------------------------------
# 2. Initialiser le tableau des contrôles
# ------------------------------------------------------------

resultats = []

def ajouter_controle(nom, resultat, attendu, statut):
    resultats.append({
        "Contrôle": nom,
        "Résultat calculé": resultat,
        "Attendu / critère": attendu,
        "Conclusion": statut
    })

def verifier_cle_dimension(table, colonne, nom):
    nb_null = int(table[colonne].isna().sum())
    nb_uniques = int(table[colonne].nunique())
    nb_lignes = len(table)

    valide = (
        nb_null == 0
        and nb_uniques == nb_lignes
    )

    ajouter_controle(
        f"Clés {nom}",
        f"{nb_uniques} uniques ; {nb_null} nulles",
        "Toutes uniques et non nulles",
        "OK" if valide else "ERREUR"
    )

    return valide

# ------------------------------------------------------------
# 3. Contrôler les clés primaires des dimensions
# ------------------------------------------------------------

print("===== 1. CONTRÔLE DES CLÉS PRIMAIRES =====")

cles_valides = []

cles_valides.append(
    verifier_cle_dimension(dim_date, "DateKey", "DimDate")
)

cles_valides.append(
    verifier_cle_dimension(dim_produit, "ProduitKey", "DimProduit")
)

cles_valides.append(
    verifier_cle_dimension(dim_client, "ClientKey", "DimClient")
)

cles_valides.append(
    verifier_cle_dimension(dim_pays, "PaysKey", "DimPays")
)

# ------------------------------------------------------------
# 4. Contrôler les clés étrangères
# ------------------------------------------------------------

print("\n===== 2. CONTRÔLE DES CLÉS ÉTRANGÈRES =====")

dimensions = [
    ("DateKey", dim_date, "DateKey"),
    ("ProduitKey", dim_produit, "ProduitKey"),
    ("ClientKey", dim_client, "ClientKey"),
    ("PaysKey", dim_pays, "PaysKey")
]

cles_etrangeres_valides = []

for cle_fait, dimension, cle_dimension in dimensions:

    valeurs_valides = set(dimension[cle_dimension].dropna())

    orphelines = (
        fact_vente[cle_fait].isna()
        | ~fact_vente[cle_fait].isin(valeurs_valides)
    )

    nombre_orphelines = int(orphelines.sum())

    valide = nombre_orphelines == 0

    cles_etrangeres_valides.append(valide)

    ajouter_controle(
        f"Clés orphelines {cle_fait}",
        nombre_orphelines,
        0,
        "OK" if valide else "ERREUR"
    )

    print(cle_fait, ":", nombre_orphelines,
          "clé(s) orpheline(s)")

# ------------------------------------------------------------
# 5. Contrôler la conservation des provenances
# ------------------------------------------------------------

print("\n===== 3. CONTRÔLE DES SOURCEINDEX =====")

source_l2 = df_prep["SourceIndex"]
source_l3 = fact_vente["SourceIndex"]

source_l2_uniques = source_l2.is_unique
source_l3_uniques = source_l3.is_unique

source_l2_set = set(source_l2.dropna())
source_l3_set = set(source_l3.dropna())

manquantes = source_l2_set - source_l3_set
supplementaires = source_l3_set - source_l2_set

ajouter_controle(
    "SourceIndex L2 uniques",
    source_l2_uniques,
    True,
    "OK" if source_l2_uniques else "ERREUR"
)

ajouter_controle(
    "SourceIndex L3 uniques",
    source_l3_uniques,
    True,
    "OK" if source_l3_uniques else "ERREUR"
)

ajouter_controle(
    "Provenances manquantes",
    len(manquantes),
    0,
    "OK" if len(manquantes) == 0 else "ERREUR"
)

ajouter_controle(
    "Provenances supplémentaires",
    len(supplementaires),
    0,
    "OK" if len(supplementaires) == 0 else "ERREUR"
)

print("Nombre de lignes L2 :", len(df_prep))
print("Nombre de lignes L3 :", len(fact_vente))
print("SourceIndex manquants dans L3 :", len(manquantes))
print("SourceIndex supplémentaires :", len(supplementaires))

# ------------------------------------------------------------
# 6. Contrôler le périmètre des ventes
# ------------------------------------------------------------

print("\n===== 4. CONTRÔLE DU PÉRIMÈTRE =====")

if "InclureFactVente" not in df_prep.columns:
    raise ValueError("Colonne InclureFactVente absente de L2")

perimetre_l2 = df_prep[
    df_prep["InclureFactVente"].eq(True)
]

source_admissible = set(
    perimetre_l2["SourceIndex"].dropna()
)

hors_perimetre = source_l3_set - source_admissible

ventes_admissibles_absentes = (
    source_admissible - source_l3_set
)

ajouter_controle(
    "Lignes hors périmètre",
    len(hors_perimetre),
    0,
    "OK" if len(hors_perimetre) == 0 else "ERREUR"
)

ajouter_controle(
    "Ventes admissibles absentes",
    len(ventes_admissibles_absentes),
    0,
    "OK" if len(ventes_admissibles_absentes) == 0 else "ERREUR"
)

# ------------------------------------------------------------
# 7. Contrôler la conservation des clients inconnus
# ------------------------------------------------------------

print("\n===== 5. CONTRÔLE CLIENT INCONNU =====")

ventes_sans_client_l2 = perimetre_l2[
    perimetre_l2["CustomerID_norm"].isna()
]

source_inconnu_l2 = set(
    ventes_sans_client_l2["SourceIndex"]
)

ventes_inconnues_l3 = fact_vente[
    fact_vente["ClientKey"].eq(0)
]

source_inconnu_l3 = set(
    ventes_inconnues_l3["SourceIndex"]
)

inconnus_perdus = source_inconnu_l2 - source_inconnu_l3
inconnus_ajoutes = source_inconnu_l3 - source_inconnu_l2

conservation_inconnus = (
    source_inconnu_l2 == source_inconnu_l3
)

ajouter_controle(
    "Ventes au client inconnu",
    len(source_inconnu_l3),
    len(source_inconnu_l2),
    "OK" if conservation_inconnus else "ERREUR"
)

ajouter_controle(
    "Provenances inconnues perdues",
    len(inconnus_perdus),
    0,
    "OK" if len(inconnus_perdus) == 0 else "ERREUR"
)

ajouter_controle(
    "Provenances inconnues ajoutées",
    len(inconnus_ajoutes),
    0,
    "OK" if len(inconnus_ajoutes) == 0 else "ERREUR"
)

# ------------------------------------------------------------
# 8. Comparer les quantités et montants ligne par ligne
# ------------------------------------------------------------

print("\n===== 6. RÉCONCILIATION DES MESURES =====")

# Reconstituer la référence L2 uniquement sur les ventes admissibles

reference_l2 = perimetre_l2[
    ["SourceIndex", "Quantity_num",
     "UnitPrice_num", "MontantLigne"]
].copy()

reference_l2 = reference_l2.rename(columns={
    "Quantity_num": "Quantite_L2",
    "UnitPrice_num": "Prix_L2",
    "MontantLigne": "Montant_L2"
})

comparaison = fact_vente.merge(
    reference_l2,
    on="SourceIndex",
    how="outer",
    indicator=True,
    validate="one_to_one"
)

print("Nombre de lignes comparables :",
      (comparaison["_merge"] == "both").sum())

# Vérification des quantités

comparaison["Ecart_Quantite"] = (
    comparaison["QuantiteVendue"]
    - comparaison["Quantite_L2"]
)

erreurs_quantite = (
    comparaison["Ecart_Quantite"].notna()
    & ~np.isclose(
        comparaison["QuantiteVendue"].fillna(0),
        comparaison["Quantite_L2"].fillna(0),
        rtol=0,
        atol=0
    )
)

# Vérification des prix unitaires

erreurs_prix = ~np.isclose(
    comparaison["PrixUnitaire"].fillna(0),
    comparaison["Prix_L2"].fillna(0),
    rtol=1e-12,
    atol=1e-12
)

# Vérification des montants

erreurs_montant = ~np.isclose(
    comparaison["MontantLigne"].fillna(0),
    comparaison["Montant_L2"].fillna(0),
    rtol=1e-12,
    atol=1e-8
)

# Comparaison des montants calculés et théoriques

montant_theorique = (
    fact_vente["QuantiteVendue"]
    * fact_vente["PrixUnitaire"]
)

montant_ligne_incoherent = ~np.isclose(
    fact_vente["MontantLigne"],
    montant_theorique,
    rtol=1e-12,
    atol=1e-8
)

ajouter_controle(
    "Écarts de quantité ligne par ligne",
    int(erreurs_quantite.sum()),
    0,
    "OK" if not erreurs_quantite.any() else "ERREUR"
)

ajouter_controle(
    "Écarts de prix unitaire",
    int(erreurs_prix.sum()),
    0,
    "OK" if not erreurs_prix.any() else "ERREUR"
)

ajouter_controle(
    "Écarts de montant ligne par ligne",
    int(erreurs_montant.sum()),
    0,
    "OK" if not erreurs_montant.any() else "ERREUR"
)

ajouter_controle(
    "Incohérences Quantité × Prix",
    int(montant_ligne_incoherent.sum()),
    0,
    "OK" if not montant_ligne_incoherent.any() else "ERREUR"
)

# ------------------------------------------------------------
# 9. Réconciliation des totaux
# ------------------------------------------------------------

print("\n===== 7. COMPARAISON DES TOTAUX =====")

quantite_l2 = reference_l2["Quantite_L2"].sum()
quantite_l3 = fact_vente["QuantiteVendue"].sum()

montant_l2 = reference_l2["Montant_L2"].sum()
montant_l3 = fact_vente["MontantLigne"].sum()

ajouter_controle(
    "Quantité totale",
    f"L2={quantite_l2} ; L3={quantite_l3}",
    "Totaux identiques",
    "OK" if np.isclose(
        quantite_l2, quantite_l3, rtol=0, atol=0
    ) else "ERREUR"
)

ajouter_controle(
    "Montant total",
    f"L2={montant_l2:.2f} ; L3={montant_l3:.2f}",
    "Totaux identiques",
    "OK" if np.isclose(
        montant_l2, montant_l3, rtol=1e-12, atol=1e-8
    ) else "ERREUR"
)

print("Quantité totale L2 :", quantite_l2)
print("Quantité totale L3 :", quantite_l3)

print("Montant total L2 :", round(montant_l2, 2))
print("Montant total L3 :", round(montant_l3, 2))

# ------------------------------------------------------------
# 10. Vérifier les montants des ventes au client inconnu
# ------------------------------------------------------------

montant_inconnu_l2 = (
    ventes_sans_client_l2["MontantLigne"].sum()
)

montant_inconnu_l3 = (
    ventes_inconnues_l3["MontantLigne"].sum()
)

ajouter_controle(
    "Montant des ventes client inconnu",
    f"L2={montant_inconnu_l2:.2f} ; L3={montant_inconnu_l3:.2f}",
    "Montants identiques",
    "OK" if np.isclose(
        montant_inconnu_l2,
        montant_inconnu_l3,
        rtol=1e-12,
        atol=1e-8
    ) else "ERREUR"
)

# ------------------------------------------------------------
# 11. Construire controles_l3
# ------------------------------------------------------------

controles_l3 = pd.DataFrame(resultats)

print("\n===== TABLEAU FINAL DES CONTRÔLES L3 =====")

display(controles_l3)

# ------------------------------------------------------------
# 12. Afficher les anomalies éventuelles
# ------------------------------------------------------------

print("\n===== ANOMALIES À EXAMINER =====")

anomalies = controles_l3[
    controles_l3["Conclusion"] == "ERREUR"
]

display(anomalies)

print("Nombre de contrôles en erreur :", len(anomalies))

# Afficher les lignes dont la provenance ne correspond pas

print("\nProvenances manquantes :")
print(list(manquantes)[:20])

print("\nProvenances supplémentaires :")
print(list(supplementaires)[:20])

print("\nÉcarts de quantité :")
display(comparaison.loc[erreurs_quantite].head(10))

print("\nÉcarts de prix :")
display(comparaison.loc[erreurs_prix].head(10))

print("\nÉcarts de montant :")
display(comparaison.loc[erreurs_montant].head(10))

# ------------------------------------------------------------
# 13. Décision finale
# ------------------------------------------------------------

if len(anomalies) == 0:
    print("\nVALIDATION : tous les contrôles L3 sont réussis.")
    print("Le modèle décisionnel peut être accepté.")
else:
    print("\nATTENTION : L3 présente des anomalies.")
    print("Corriger les erreurs avant l'acceptation.")

print("\nÉTAPE 11 TERMINÉE.")

===== 1. CONTRÔLE DES CLÉS PRIMAIRES =====

===== 2. CONTRÔLE DES CLÉS ÉTRANGÈRES =====
DateKey : 0 clé(s) orpheline(s)
ProduitKey : 0 clé(s) orpheline(s)
ClientKey : 0 clé(s) orpheline(s)
PaysKey : 0 clé(s) orpheline(s)

===== 3. CONTRÔLE DES SOURCEINDEX =====
Nombre de lignes L2 : 522364
Nombre de lignes L3 : 522364
SourceIndex manquants dans L3 : 0
SourceIndex supplémentaires : 0

===== 4. CONTRÔLE DU PÉRIMÈTRE =====

===== 5. CONTRÔLE CLIENT INCONNU =====

===== 6. RÉCONCILIATION DES MESURES =====
Nombre de lignes comparables : 522364

===== 7. COMPARAISON DES TOTAUX =====
Quantité totale L2 : 5561117
Quantité totale L3 : 5561117
Montant total L2 : 10244661.18
Montant total L3 : 10244661.18

===== TABLEAU FINAL DES CONTRÔLES L3 =====


,Contrôle,Résultat calculé,Attendu / critère,Conclusion
0,Clés DimDate,305 uniques ; 0 nulles,Toutes uniques et non nulles,OK
1,Clés DimProduit,4026 uniques ; 0 nulles,Toutes uniques et non nulles,OK
2,Clés DimClient,4335 uniques ; 0 nulles,Toutes uniques et non nulles,OK
3,Clés DimPays,38 uniques ; 0 nulles,Toutes uniques et non nulles,OK
4,Clés orphelines DateKey,0,0,OK
5,Clés orphelines ProduitKey,0,0,OK
6,Clés orphelines ClientKey,0,0,OK
7,Clés orphelines PaysKey,0,0,OK
8,SourceIndex L2 uniques,True,True,OK
9,SourceIndex L3 uniques,True,True,OK



===== ANOMALIES À EXAMINER =====


,Contrôle,Résultat calculé,Attendu / critère,Conclusion


Nombre de contrôles en erreur : 0

Provenances manquantes :
[]

Provenances supplémentaires :
[]

Écarts de quantité :


,FactVenteKey,DateKey,ProduitKey,ClientKey,PaysKey,InvoiceNo,QuantiteVendue,PrixUnitaire,MontantLigne,SourceIndex,SourceFile,SourceRow,InvoiceDate_horodatage,Quantite_L2,Prix_L2,Montant_L2,_merge,Ecart_Quantite



Écarts de prix :


,FactVenteKey,DateKey,ProduitKey,ClientKey,PaysKey,InvoiceNo,QuantiteVendue,PrixUnitaire,MontantLigne,SourceIndex,SourceFile,SourceRow,InvoiceDate_horodatage,Quantite_L2,Prix_L2,Montant_L2,_merge,Ecart_Quantite



Écarts de montant :


,FactVenteKey,DateKey,ProduitKey,ClientKey,PaysKey,InvoiceNo,QuantiteVendue,PrixUnitaire,MontantLigne,SourceIndex,SourceFile,SourceRow,InvoiceDate_horodatage,Quantite_L2,Prix_L2,Montant_L2,_merge,Ecart_Quantite



VALIDATION : tous les contrôles L3 sont réussis.
Le modèle décisionnel peut être accepté.

ÉTAPE 11 TERMINÉE.


### Preuve de l’étape — à renseigner après exécution
- **Variables ou tables obtenues :** La table controles_l3 contenant les résultats des contrôles d’intégrité et de réconciliation. Les tables FactVente, DimDate, DimProduit, DimClient et DimPays ont également été vérifiées.
- **Résultat du contrôle / observations :** -  Tous les contrôles sont réussis (0 anomalie). Les clés primaires sont uniques et non nulles, aucune clé étrangère orpheline n’a été détectée et les 522 364 ventes de L2 sont conservées dans L3. Les quantités et les montants sont identiques entre les deux niveaux :
- Quantité totale : 5 561 117.
- Montant total : 10 244 661,18.
- Ventes rattachées au client inconnu : 131 355.
- Montant des ventes au client inconnu : 1 509 642,11.

Aucun écart n’a été constaté sur les quantités, les prix unitaires ou les montants ligne par ligne.
- **Décision ou correction :** Aucune correction n’est nécessaire. Toutes les ventes ont été conservées, les relations entre les tables sont valides et les mesures sont correctement réconciliées. Le modèle décisionnel L3 est donc validé et peut être accepté pour la suite du TP.

# ÉTAPE 12 — Justifier l’architecture : étoile, flocon, entrepôt et mart
**Durée indicative : 10 min · Objectif :** Défendre l’organisation choisie sans construire un second modèle.



## Questions du TP
1. Pourquoi l’étoile est-elle adaptée au MVP ?

## Question complémentaire — prendre du recul
Une hiérarchie Jour → Mois → Trimestre → Année oblige-t-elle à créer une table par niveau ? Séparer une catégorie d’une dimension changerait-il le nombre de ventes ? Peut-on le faire ici sans source de catégorie ?

### Réponses du binôme — à compléter

**Architecture retenue et deux arguments liés au projet :** Architecture en étoile. Elle est simple à comprendre et facilite les analyses des ventes grâce aux relations directes entre le fait et les dimensions.  
**Différence entre organisation des tables et grain des faits :**  L'organisation des tables décrit la manière dont les dimensions sont reliées au fait. Le grain définit ce que représente une ligne de FactVente : une ligne de vente d'un produit sur une facture.  
**Différence entre hiérarchie et flocon :** Une hiérarchie organise les niveaux d'analyse dans une même dimension. Le flocon consiste à séparer certaines informations dans plusieurs tables reliées entre elles.  
**Où se situe notre Data Mart Ventes dans le DW minimal ?** Le Data Mart Ventes correspond à la partie du Data Warehouse consacrée à l'analyse des ventes. Il utilise la table FactVente et ses quatre dimensions.  
**Pourquoi ne construit-on pas un second modèle en flocon ici ?** Parce que l'architecture en étoile répond déjà aux besoins du MVP. Un modèle en flocon ajouterait des tables et des jointures sans apporter de bénéfice nécessaire au projet.

**Notre justification :**  
Nous avons retenu une architecture en étoile composée d'une table de faits et de quatre dimensions. Ce choix permet d'analyser les ventes selon plusieurs axes : le temps, les produits, les clients et les pays. Le modèle respecte le grain défini, conserve toutes les ventes et garantit l'intégrité des relations. Nous n'avons pas ajouté de dimensions ou d'attributs non présents dans les sources.

**Point à vérifier / difficulté rencontrée :**  
Vérifier que chaque dimension possède une clé primaire unique, que toutes les clés étrangères de FactVente correspondent à une dimension existante et que les hiérarchies temporelles sont correctement définies. Aucune difficulté majeure n'a été constatée lors de la validation du modèle.

In [16]:
import pandas as pd

# ============================================================
# ÉTAPE 12 — JUSTIFIER L'ARCHITECTURE : ÉTOILE, DW ET DATA MART
# ============================================================

# 1. DICTIONNAIRE DU MODELE : GRAIN, CLES ET ATTRIBUTS

modele_final = [
    {
        "Table": "FactVente",
        "Type": "Table de faits",
        "Grain": "Une ligne de vente d'un produit sur une facture",
        "Cle primaire": "FactVenteKey",
        "Cles etrangeres": "DateKey, ProduitKey, ClientKey, PaysKey",
        "Attributs / mesures": "InvoiceNo, Quantity, UnitPrice, MontantLigne, SourceIndex"
    },
    {
        "Table": "DimDate",
        "Type": "Dimension",
        "Grain": "Un jour du calendrier",
        "Cle primaire": "DateKey",
        "Cles etrangeres": "Aucune",
        "Attributs / mesures": "Date, Jour, NumMois, Mois, Trimestre, Annee, AnneeMois"
    },
    {
        "Table": "DimProduit",
        "Type": "Dimension",
        "Grain": "Un couple StockCode / Description normalisée",
        "Cle primaire": "ProduitKey",
        "Cles etrangeres": "Aucune",
        "Attributs / mesures": "StockCode_norm, Description_norm"
    },
    {
        "Table": "DimClient",
        "Type": "Dimension",
        "Grain": "Un client identifié ou le client inconnu",
        "Cle primaire": "ClientKey",
        "Cles etrangeres": "Aucune",
        "Attributs / mesures": "ClientID_norm, StatutIdentification"
    },
    {
        "Table": "DimPays",
        "Type": "Dimension",
        "Grain": "Un pays",
        "Cle primaire": "PaysKey",
        "Cles etrangeres": "Aucune",
        "Attributs / mesures": "Country_norm"
    }
]

df_modele_final = pd.DataFrame(modele_final)

print("1. CONTRAT DU MODELE DECISIONNEL")
display(df_modele_final)


# 2. SCHEMA FINAL EN ETOILE

print("\n2. SCHEMA FINAL EN ETOILE")

schema = """
                         DIMDATE
                    (DateKey)
                         |
                         | 1
                         |
                         | N
DIMPRODUIT 1 -------- FACTVENTE -------- 1 DIMCLIENT
(ProduitKey) N       (FactVenteKey)      N (ClientKey)
                         |
                         | N
                         |
                         | 1
                       DIMPAYS
                     (PaysKey)
"""

print(schema)


# 3. HIERARCHIE TEMPORELLE

print("\n3. HIERARCHIE TEMPORELLE")

hierarchie_date = {
    "Niveau 1": "Annee",
    "Niveau 2": "Trimestre",
    "Niveau 3": "Mois / AnneeMois",
    "Niveau 4": "Jour / Date"
}

df_hierarchie = pd.DataFrame(
    list(hierarchie_date.items()),
    columns=["Niveau", "Attribut"]
)

display(df_hierarchie)

print("La hierarchie est contenue dans une seule dimension : DimDate.")


# 4. JUSTIFICATION DE L'ARCHITECTURE

print("\n4. JUSTIFICATION DE L'ARCHITECTURE")

justification = pd.DataFrame([
    ["Architecture retenue", "Etoile"],
    ["Table de faits", "FactVente"],
    ["Nombre de dimensions", 4],
    ["Grain du fait", "Une ligne de vente d'un produit sur une facture"],
    ["Data Mart", "Data Mart Ventes"],
    ["Hierarchie temporelle", "Annee > Trimestre > Mois > Jour"],
    ["Table categorie", "Non créée : aucune source fiable disponible"],
    ["Second modele en flocon", "Non nécessaire pour le MVP"]
], columns=["Element", "Description"])

display(justification)


# 5. CONTROLES DE STRUCTURE DU MODELE

print("\n5. CONTROLES DE STRUCTURE")

assert len(modele_final) == 5
assert df_modele_final["Table"].is_unique

print("Nombre de tables :", len(modele_final))
print("Nombre de dimensions : 4")
print("Nombre de tables de faits : 1")
print("Noms des tables uniques :", df_modele_final["Table"].is_unique)

print("\n6. DECISION FINALE")
print("Architecture en etoile retenue et documentee.")
print("Le Data Mart Ventes est constitue autour de FactVente.")
print("Aucune dimension non justifiee par les sources n'a ete ajoutee.")

1. CONTRAT DU MODELE DECISIONNEL


,Table,Type,Grain,Cle primaire,Cles etrangeres,Attributs / mesures
0,FactVente,Table de faits,Une ligne de vente d'un produit sur une facture,FactVenteKey,"DateKey, ProduitKey, ClientKey, PaysKey","InvoiceNo, Quantity, UnitPrice, MontantLigne, ..."
1,DimDate,Dimension,Un jour du calendrier,DateKey,Aucune,"Date, Jour, NumMois, Mois, Trimestre, Annee, A..."
2,DimProduit,Dimension,Un couple StockCode / Description normalisée,ProduitKey,Aucune,"StockCode_norm, Description_norm"
3,DimClient,Dimension,Un client identifié ou le client inconnu,ClientKey,Aucune,"ClientID_norm, StatutIdentification"
4,DimPays,Dimension,Un pays,PaysKey,Aucune,Country_norm



2. SCHEMA FINAL EN ETOILE

                         DIMDATE
                    (DateKey)
                         |
                         | 1
                         |
                         | N
DIMPRODUIT 1 -------- FACTVENTE -------- 1 DIMCLIENT
(ProduitKey) N       (FactVenteKey)      N (ClientKey)
                         |
                         | N
                         |
                         | 1
                       DIMPAYS
                     (PaysKey)


3. HIERARCHIE TEMPORELLE


,Niveau,Attribut
0,Niveau 1,Annee
1,Niveau 2,Trimestre
2,Niveau 3,Mois / AnneeMois
3,Niveau 4,Jour / Date


La hierarchie est contenue dans une seule dimension : DimDate.

4. JUSTIFICATION DE L'ARCHITECTURE


,Element,Description
0,Architecture retenue,Etoile
1,Table de faits,FactVente
2,Nombre de dimensions,4
3,Grain du fait,Une ligne de vente d'un produit sur une facture
4,Data Mart,Data Mart Ventes
5,Hierarchie temporelle,Annee > Trimestre > Mois > Jour
6,Table categorie,Non créée : aucune source fiable disponible
7,Second modele en flocon,Non nécessaire pour le MVP



5. CONTROLES DE STRUCTURE
Nombre de tables : 5
Nombre de dimensions : 4
Nombre de tables de faits : 1
Noms des tables uniques : True

6. DECISION FINALE
Architecture en etoile retenue et documentee.
Le Data Mart Ventes est constitue autour de FactVente.
Aucune dimension non justifiee par les sources n'a ete ajoutee.


### Preuve de l’étape — à renseigner après exécution
- **Variables ou tables obtenues :** La variable df_modele_final contenant le contrat du modèle décisionnel, df_hierarchie décrivant la hiérarchie temporelle et justification présentant les choix d'architecture. Le schéma final en étoile a également été affiché.
- **Résultat du contrôle / observations :** - Le modèle contient 5 tables : une table de faits (FactVente) et 4 dimensions (DimDate, DimProduit, DimClient et DimPays). Les noms des tables sont uniques (True). Le grain de la table de faits est défini comme une ligne de vente d'un produit sur une facture. La hiérarchie temporelle est organisée selon l'ordre Année → Trimestre → Mois → Jour. Aucune catégorie ou dimension supplémentaire non justifiée par les sources n'a été ajoutée.
- **Décision ou correction :** L'architecture en étoile est retenue et validée pour le MVP. Le Data Mart Ventes est constitué autour de FactVente et de ses quatre dimensions. Aucun second modèle en flocon n'est nécessaire, car l'organisation actuelle répond aux besoins d'analyse des ventes et reste simple à exploiter. Aucune correction n'est nécessaire.

# ÉTAPE 13 — Défi technique : détecter une rupture d’intégrité
**Durée indicative : 10 min · Objectif :** Diagnostiquer une erreur sans supprimer une vente légitime.



## Questions du TP
1. Écrivez un contrôle qui identifie les clés de FactVente absentes d’une dimension, puis expliquez comment corriger le rattachement sans supprimer la ligne de vente.

## Question complémentaire — prendre du recul
Choisissez une erreur : référence produit dupliquée, référence absente, client inconnu perdu ou DateKey répétée. Quel contrôle doit échouer ? Le montant final pourrait-il rester correct malgré l’erreur ?

### Réponses du binôme — à compléter

**Erreur choisie et prédiction avant le test :** Nous choisissons une référence produit absente de DimProduit. Nous prévoyons que le contrôle des clés étrangères détectera une clé ProduitKey qui ne correspond à aucune clé primaire de la dimension.  
**Contrôle déclenché / sortie observée :** Le contrôle compare les ProduitKey de FactVente avec les clés primaires de DimProduit. Une clé absente est détectée comme une clé orpheline. 
**Cause et correction proposée :** L'erreur peut être causée par un échec de rapprochement entre les ventes et la dimension produit. Pour corriger le problème, il faut vérifier les colonnes StockCode_norm et Description_norm, puis refaire le rattachement. Si le produit est réellement absent, on peut ajouter un membre technique « Produit inconnu » dans DimProduit et rattacher la vente à cette clé, sans supprimer la vente. 
**Preuve après correction :** Après correction, le nombre de clés orphelines doit être égal à zéro. La vente doit toujours être présente dans FactVente.

**Notre justification :**  
Le contrôle de l'intégrité référentielle garantit que chaque vente possède une référence valide vers ses dimensions. Une erreur de rattachement ne doit jamais entraîner la suppression d'une vente légitime. Il faut corriger la relation tout en conservant la provenance et les mesures de la vente.

**Point à vérifier / difficulté rencontrée :**  
Vérifier que la correction ne modifie ni le nombre de ventes, ni les quantités, ni les montants. Il faut également vérifier que les tables originales validées ne sont pas modifiées pendant le test.

In [17]:
# ============================================================
# ETAPE 13 — DETECTER UNE RUPTURE D'INTEGRITE
# Erreur choisie : reference produit absente
# ============================================================

import pandas as pd

# 1. CREER DES COPIES DE DEMONSTRATION
fact_test = fact_vente.copy()
produit_test = dim_produit.copy()

print("1. ETAT INITIAL")
print("Nombre de ventes :", len(fact_test))
print("Nombre de produits :", len(produit_test))

# 2. INTRODUIRE UNE ERREUR CONTROLEE
# On modifie une seule clé produit dans la copie du fait.
# Les tables originales ne sont pas modifiees.

ancienne_cle = fact_test.loc[0, "ProduitKey"]
cle_invalide = -999

fact_test.loc[0, "ProduitKey"] = cle_invalide

print("\n2. ERREUR INTRODUITE")
print("Ancienne ProduitKey :", ancienne_cle)
print("Nouvelle ProduitKey :", cle_invalide)

# 3. DEFINIR LE CONTROLE D'INTEGRITE

def verifier_cles_etrangeres(fait, dimension, colonne):
    cles_valides = set(dimension[colonne].dropna())
    cles_utilisees = set(fait[colonne].dropna())

    cles_orphelines = cles_utilisees - cles_valides

    lignes_orphelines = fait[
        ~fait[colonne].isin(cles_valides)
    ]

    return cles_orphelines, lignes_orphelines


# 4. DETECTER L'ERREUR

orphelines_avant, lignes_avant = verifier_cles_etrangeres(
    fact_test,
    produit_test,
    "ProduitKey"
)

print("\n3. RESULTAT DU CONTROLE AVANT CORRECTION")
print("Cles produit orphelines :", orphelines_avant)
print("Nombre de lignes concernees :", len(lignes_avant))

# 5. CORRIGER LE RATTACHEMENT
# On restaure ici la vraie clé connue avant l'erreur.
# Dans un cas réel, il faut refaire le lookup sur le couple
# StockCode_norm + Description_norm.

fact_test.loc[0, "ProduitKey"] = ancienne_cle

print("\n4. CORRECTION EFFECTUEE")
print("ProduitKey restauree :", fact_test.loc[0, "ProduitKey"])

# 6. REJOUER LE CONTROLE

orphelines_apres, lignes_apres = verifier_cles_etrangeres(
    fact_test,
    produit_test,
    "ProduitKey"
)

print("\n5. CONTROLE APRES CORRECTION")
print("Cles produit orphelines :", orphelines_apres)
print("Nombre de lignes concernees :", len(lignes_apres))

# 7. VERIFIER LA CONSERVATION DES VENTES

print("\n6. CONTROLE DE CONSERVATION")

print("Nombre de ventes initial :", len(fact_vente))
print("Nombre de ventes apres correction :", len(fact_test))

assert len(fact_test) == len(fact_vente)
assert len(orphelines_apres) == 0

print("Nombre de ventes conservees :", len(fact_test))
print("Integrite referentielle : OK")

# 8. VERIFIER QUE LES TABLES ORIGINALES SONT INTACTES

assert fact_vente.loc[0, "ProduitKey"] == ancienne_cle

print("\n7. VALIDATION FINALE")
print("Les tables originales sont intactes.")
print("Aucune vente n'a ete supprimee.")
print("La reference produit est valide apres correction.")
print("ETAPE 13 TERMINEE.")

1. ETAT INITIAL
Nombre de ventes : 522364
Nombre de produits : 4026

2. ERREUR INTRODUITE
Ancienne ProduitKey : 3533
Nouvelle ProduitKey : -999

3. RESULTAT DU CONTROLE AVANT CORRECTION
Cles produit orphelines : {-999}
Nombre de lignes concernees : 1

4. CORRECTION EFFECTUEE
ProduitKey restauree : 3533

5. CONTROLE APRES CORRECTION
Cles produit orphelines : set()
Nombre de lignes concernees : 0

6. CONTROLE DE CONSERVATION
Nombre de ventes initial : 522364
Nombre de ventes apres correction : 522364
Nombre de ventes conservees : 522364
Integrite referentielle : OK

7. VALIDATION FINALE
Les tables originales sont intactes.
Aucune vente n'a ete supprimee.
La reference produit est valide apres correction.
ETAPE 13 TERMINEE.


### Preuve de l’étape — à renseigner après exécution
- **Variables ou tables obtenues :** Une copie de démonstration de FactVente a été créée pour introduire volontairement une erreur sur ProduitKey. La clé -999 a été utilisée pour simuler une référence produit inexistante.
- **Résultat du contrôle / observations :** Le contrôle a détecté une clé produit orpheline (-999) concernant une seule vente. Après correction, la clé ProduitKey a été restaurée à sa valeur initiale (3533). Le contrôle ne détecte alors plus aucune clé orpheline. Les 522 364 ventes ont été conservées.
- **Décision ou correction :**  La clé étrangère incorrecte a été corrigée en rétablissant la référence produit valide. L'intégrité référentielle est rétablie. Aucune vente n'a été supprimée et les tables originales sont restées intactes.

# ÉTAPE 14 — Exporter L3 et préparer le prochain TP OLAP
**Durée indicative : 15 min · Objectif :** Remettre un modèle alimenté et documenté, réutilisable.



## Questions du TP
1. Remettre la phrase de grain, le schéma en étoile, les tables DimDate/DimProduit/DimClient/DimPays et FactVente, le script ou notebook d’alimentation et les contrôles CP3–CP5.

## Question complémentaire — prendre du recul
Pour le prochain TP sur l’OLAP, proposer deux questions d’exploration possibles à partir de L3. Pour chacune : mesure, axes, niveau temporel, périmètre et contrôle de référence. Quelle question ne pouvez-vous pas traiter avec les attributs réellement disponibles ?

### Réponses du binôme — à compléter

| Pièce L3 | Nom de fichier / emplacement | Contrôle avant remise |
| --- | --- | --- |
| DimDate | DimDate.csv | DateKey unique et non nulle |
| DimProduit | DimProduit.csv | ProduitKey unique et non nul |
| DimClient | DimClient.csv | ClientKey unique et présence du client inconnu |
| DimPays | DimPays.csv | PaysKey unique et non nul |
| FactVente | FactVente.csv | FactVenteKey unique et clés étrangères valides |
| Grain, schéma et contrôles | Documentation_L3.txt | Grain et relations documentés |
| Notebook exécuté | TP_Entrepot_Ventes.ipynb | Toutes les étapes exécutées sans erreur |

| Question pour le prochain TP OLAP | Mesure | Axes / niveau | Périmètre et contrôle |
| --- | --- | --- | --- |
| Comment évolue le chiffre d'affaires selon les années et les pays ? | Somme de MontantLigne | Pays, Année, Trimestre | Toutes les ventes de L3 ; comparer le total au montant global de FactVente |
| Quels sont les produits les plus vendus et ceux qui génèrent le plus de chiffre d'affaires ? | Somme de Quantity et somme de MontantLigne | Produit, Année, Mois | Toutes les ventes de L3 ; vérifier les totaux des quantités et des montants |

**Une question non traitable et l’information manquante :** Nous ne pouvons pas analyser le chiffre d'affaires par catégorie ou par marque de produit, car ces informations ne sont pas disponibles dans les fichiers sources.
Il faudrait disposer d'une source fiable contenant les catégories et les marques des produits pour réaliser cette analyse.

**Notre justification :**  
Nous avons construit un Data Mart Ventes organisé en étoile, composé d'une table de faits FactVente et de quatre dimensions : DimDate, DimProduit, DimClient et DimPays.
Les tables sont exportées séparément au format CSV afin de faciliter leur réutilisation dans le prochain TP OLAP. Les contrôles d'intégrité, de conservation des ventes et de réconciliation des montants ont été réalisés avant l'export.

**Point à vérifier / difficulté rencontrée :**  
Vérifier que les cinq fichiers CSV sont bien générés, que les clés primaires sont uniques, que les clés étrangères sont valides et que les données exportées correspondent aux tables L3 validées.

In [19]:
# ============================================================
# ETAPE 14 : CONTROLE DE LA QUALITE DU DATA WAREHOUSE
# ============================================================

import pandas as pd

# ============================================================
# 1. CONTROLE DES CLES PRIMAIRES
# ============================================================

print("=" * 60)
print("1. CONTROLE DES CLES PRIMAIRES")
print("=" * 60)

dimensions = {
    "DimDate": dim_date,
    "DimProduit": dim_produit,
    "DimClient": dim_client,
    "DimPays": dim_pays,
    "FactVente": fact_vente
}

cles_primaires = {
    "DimDate": "DateKey",
    "DimProduit": "ProduitKey",
    "DimClient": "ClientKey",
    "DimPays": "PaysKey",
    "FactVente": "VenteKey"
}

for nom, df in dimensions.items():
    cle = cles_primaires[nom]

    if cle not in df.columns:
        print(f"{nom} : colonne {cle} introuvable")
        continue

    uniques = df[cle].nunique()
    total = len(df)

    if df[cle].isnull().any():
        print(f"{nom} : clés nulles - ERREUR")
    elif uniques != total:
        print(f"{nom} : clés dupliquées - ERREUR")
    else:
        print(f"{nom} : {uniques} clés uniques - OK")


# ============================================================
# 2. CONTROLE DES CLES ETRANGERES
# ============================================================

print("\n" + "=" * 60)
print("2. CONTROLE DES CLES ETRANGERES")
print("=" * 60)

cles_etrangeres = {
    "DateKey": (dim_date, "DateKey"),
    "ProduitKey": (dim_produit, "ProduitKey"),
    "ClientKey": (dim_client, "ClientKey"),
    "PaysKey": (dim_pays, "PaysKey")
}

for cle, (dimension, cle_dimension) in cles_etrangeres.items():

    if cle not in fact_vente.columns:
        print(f"{cle} : colonne introuvable")
        continue

    cles_valides = set(dimension[cle_dimension].dropna())

    orphelines = (
        fact_vente[cle].notna()
        & ~fact_vente[cle].isin(cles_valides)
    ).sum()

    print(f"{cle} : {orphelines} clé(s) orpheline(s)")


# ============================================================
# 3. CONTROLE DU GRAIN ET DE LA CONSERVATION
# ============================================================

print("\n" + "=" * 60)
print("3. CONTROLE DU GRAIN ET DE LA CONSERVATION")
print("=" * 60)

print("Nombre de ventes :", len(fact_vente))

if "SourceIndex" in fact_vente.columns:
    print(
        "SourceIndex uniques :",
        fact_vente["SourceIndex"].nunique()
    )

    print(
        "Doublons SourceIndex :",
        fact_vente["SourceIndex"].duplicated().sum()
    )

print("Grain : une ligne de vente d'un produit sur une facture.")


# ============================================================
# 4. CONTROLE DES MESURES
# ============================================================

print("\n" + "=" * 60)
print("4. CONTROLE DES MESURES")
print("=" * 60)

# Détection automatique de la colonne quantité
colonnes_quantite = [
    "Quantite",
    "Quantity_num",
    "Quantity",
    "quantity",
    "Qte"
]

col_quantite = next(
    (c for c in colonnes_quantite if c in fact_vente.columns),
    None
)

# Vérification de la colonne montant
col_montant = "MontantLigne"

if col_quantite is not None:

    quantite_totale = pd.to_numeric(
        fact_vente[col_quantite], errors="coerce"
    ).sum()

    print("Colonne quantité utilisée :", col_quantite)
    print("Quantité totale :", quantite_totale)

else:
    print("ERREUR : aucune colonne de quantité trouvée.")
    print("Colonnes disponibles :", fact_vente.columns.tolist())

if col_montant in fact_vente.columns:

    montant_total = pd.to_numeric(
        fact_vente[col_montant], errors="coerce"
    ).sum()

    print("Montant total :", montant_total)

else:
    print("ERREUR : colonne MontantLigne introuvable.")

# Contrôle des valeurs nulles et négatives
if col_quantite is not None:
    quantites = pd.to_numeric(
        fact_vente[col_quantite], errors="coerce"
    )

    print("Quantités nulles :", quantites.isna().sum())
    print("Quantités négatives :", (quantites < 0).sum())

if col_montant in fact_vente.columns:
    montants = pd.to_numeric(
        fact_vente[col_montant], errors="coerce"
    )

    print("Montants nuls :", montants.isna().sum())
    print("Montants négatifs :", (montants < 0).sum())


# ============================================================
# 5. BILAN FINAL
# ============================================================

print("\n" + "=" * 60)
print("5. BILAN FINAL")
print("=" * 60)

print("DimDate :", len(dim_date), "lignes")
print("DimProduit :", len(dim_produit), "lignes")
print("DimClient :", len(dim_client), "lignes")
print("DimPays :", len(dim_pays), "lignes")
print("FactVente :", len(fact_vente), "lignes")

print("\nCONTROLE TERMINE.")

1. CONTROLE DES CLES PRIMAIRES
DimDate : 305 clés uniques - OK
DimProduit : 4026 clés uniques - OK
DimClient : 4335 clés uniques - OK
DimPays : 38 clés uniques - OK
FactVente : colonne VenteKey introuvable

2. CONTROLE DES CLES ETRANGERES
DateKey : 0 clé(s) orpheline(s)
ProduitKey : 0 clé(s) orpheline(s)
ClientKey : 0 clé(s) orpheline(s)
PaysKey : 0 clé(s) orpheline(s)

3. CONTROLE DU GRAIN ET DE LA CONSERVATION
Nombre de ventes : 522364
SourceIndex uniques : 522364
Doublons SourceIndex : 0
Grain : une ligne de vente d'un produit sur une facture.

4. CONTROLE DES MESURES
ERREUR : aucune colonne de quantité trouvée.
Colonnes disponibles : ['FactVenteKey', 'DateKey', 'ProduitKey', 'ClientKey', 'PaysKey', 'InvoiceNo', 'QuantiteVendue', 'PrixUnitaire', 'MontantLigne', 'SourceIndex', 'SourceFile', 'SourceRow', 'InvoiceDate_horodatage']
Montant total : 10244661.18
Montants nuls : 0
Montants négatifs : 0

5. BILAN FINAL
DimDate : 305 lignes
DimProduit : 4026 lignes
DimClient : 4335 lignes
Dim

### Preuve de l’étape — à renseigner après exécution
- **Variables ou tables obtenues :** DimDate (305 lignes), DimProduit (4 026 lignes), DimClient (4 335 lignes), DimPays (38 lignes) et FactVente (522 364 lignes).
- **Résultat du contrôle / observations :**
- Les clés primaires des quatre dimensions sont uniques et valides.
- Les clés étrangères ne présentent aucune clé orpheline (0 anomalie).
- Les 522 364 ventes sont conservées, avec 522 364 valeurs SourceIndex uniques et aucun doublon.
- Le montant total des ventes est de *10 244 661,18*.
- Aucun montant nul ou négatif n'a été détecté.
- Deux problèmes ont été identifiés : le nom de la clé primaire de FactVente est FactVenteKey et non VenteKey, et le nom de la colonne quantité est QuantiteVendue.
- **Décision ou correction :** Corriger les noms des colonnes dans le code de contrôle en utilisant FactVenteKey pour la clé primaire et QuantiteVendue pour la quantité vendue, puis relancer l'étape 14 afin de valider définitivement tous les contrôles.

# Bilan du binôme — prêt pour l’OLAP ?

Cocher seulement après avoir produit la preuve correspondante.

- [ ] Nous pouvons expliquer le grain sans citer uniquement une clé technique.
- [ ] Nous distinguons une mesure, un identifiant et un attribut descriptif.
- [ ] Les quatre dimensions du MVP existent et leurs clés sont uniques.
- [ ] Le fait conserve InvoiceNo, les mesures et les références attendues.
- [ ] Les ventes admissibles ne sont ni perdues ni multipliées.
- [ ] Le client inconnu est conservé et le pays reste exploitable.
- [ ] Les références absentes sont diagnostiquées, pas masquées.
- [ ] Les montants sont réconciliés avec L2 et après relecture des CSV.
- [ ] Les données et commentaires permettent à un autre binôme de reproduire L3.
- [ ] Nous avons formulé deux questions pour le prochain TP **OLAP**.

**Notre principale difficulté et ce que nous avons appris :**  
…

**Point qui reste à clarifier avant le prochain TP :**  
…

## Pour aller plus loin — facultatif, après le parcours essentiel
1. Que faudrait-il ajouter pour conserver les clés lors de chargements successifs ? Pourquoi renuméroter une dimension peut-il invalider les anciennes références ?
2. Un Data Mart peut-il avoir plusieurs tables de faits ? Si ventes et états de stock partagent certaines dimensions, pourquoi faut-il préciser le grain de chacun avant de les comparer ?
3. Expliquez l’idempotence : que devrait-il se passer si le même chargement initial est relancé avec la même entrée et les mêmes règles ? Distinguez reconstruction contrôlée et ajout répété de lignes.

Ces questions sont des prolongements : ne pas construire une seconde architecture ni une alimentation incrémentielle dans ce TP.

---
**Références du support :** sujet joint `TP3_M115_GRAIN_MODELE_ETL(3).html`, fiche maître TP3, fiche maître C4 et dossier dataset final M115. Les questions de la fiche sont reprises et réparties dans la progression ; les questions complémentaires restent dans le périmètre étudié jusqu’à C4. Le piège DimDate du code de départ est traité comme diagnostic à résoudre, sans reproduire une clé au jour calculée à partir d’horodatages distincts. Le sujet original n’est pas modifié.

**Aucun corrigé ni résultat chiffré n’est prérempli.** Les cellules de code guidées sont volontairement à compléter : le notebook ne construit pas les tables tant que votre code n’est pas écrit. Les vérifications proposées doivent être effectivement exécutées avant de considérer L3 comme validé.